In [ ]:
!pip install pennylane xgboost pmdarima qiskit_aer pennylane-qiskit

In [ ]:
import torch
import numpy as np
from sklearn.metrics import recall_score

# ============================================================
# 1. Cargar dataset y extraer dana_te / p90_te
# ============================================================
loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
ds = next((e for e in loaded if hasattr(e, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded

dana_te = np.asarray(ds.dana_te).astype(bool)
p90_te = ds.y_te_ext.numpy().astype(bool)

print(f"Casos DANA en test set: {dana_te.sum()} de {len(dana_te)}")
print(f"Casos P90 en test set: {p90_te.sum()} de {len(p90_te)}")


# ============================================================
# 2. Extraer probabilidades predichas de cada modelo (mismo patron
#    que usamos para PR-AUC hace varias rondas)
# ============================================================
def extraer_probs(res: dict) -> dict:
    out = dict(res.get('_probs', {}))
    for k, v in res.items():
        if k.startswith('_'):
            continue
        if isinstance(v, dict) and '_prob' in v:
            out[k] = v['_prob']
    return out

checkpoints = ['02_baselines.pt', '03_qml.pt', '04_qksvm.pt']
probs_todos = {}
for ckpt in checkpoints:
    try:
        res = torch.load(f'/content/resultados/checkpoints/{ckpt}', weights_only=False)
        probs_todos.update(extraer_probs(res))
    except FileNotFoundError:
        print(f"[!] {ckpt} no encontrado, saltando.")

print(f"\nModelos disponibles: {list(probs_todos.keys())}")


# ============================================================
# 3. Wilson CI + Rc-DANA / Rc-P90 con intervalo, para cada modelo
# ============================================================
def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2*n)) / denom
    margin = z * np.sqrt((p_hat*(1-p_hat) + z**2/(4*n)) / n) / denom
    return (max(0, center - margin), min(1, center + margin))


print(f"\n{'Modelo':<20}{'Rc-DANA':>12}{'IC95% DANA':>22}{'Rc-P90':>12}{'IC95% P90':>22}")
print("-" * 90)
for nombre, prob in sorted(probs_todos.items()):
    pred = (np.asarray(prob) >= 0.5).astype(float)

    aciertos_dana = pred[dana_te].sum()
    n_dana = dana_te.sum()
    rc_dana = aciertos_dana / n_dana if n_dana > 0 else np.nan
    ci_dana = wilson_ci(aciertos_dana, n_dana)

    aciertos_p90 = pred[p90_te].sum()
    n_p90 = p90_te.sum()
    rc_p90 = aciertos_p90 / n_p90 if n_p90 > 0 else np.nan
    ci_p90 = wilson_ci(aciertos_p90, n_p90)

    print(f"{nombre:<20}{rc_dana:>12.3f}  [{ci_dana[0]:.3f}, {ci_dana[1]:.3f}]"
          f"{rc_p90:>12.3f}  [{ci_p90[0]:.3f}, {ci_p90[1]:.3f}]")

In [ ]:
import torch
import numpy as np
from sklearn.metrics import recall_score

# ============================================================
# 1. Cargar dataset y extraer dana_te / p90_te
# ============================================================
loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
ds = next((e for e in loaded if hasattr(e, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded

print(f"P60: {ds.umbral}")
print(f"P90: {ds.umbral_ext}")
print(f"DANA total en el panel: {int(ds.dana_all.sum()) if hasattr(ds, 'dana_all') else 'no disponible'}")

In [ ]:
import torch

loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
ds = next((e for e in loaded if hasattr(e, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded

print("=== VERIFICACION DEFINITIVA DEL PIPELINE CANONICO ===")
print(f"P60 (ds.umbral):     {ds.umbral}")
print(f"P90 (ds.umbral_ext): {ds.umbral_ext}")
print(f"DANA total en panel (ds.dana_all.sum() si existe): "
      f"{int(ds.dana_all.sum()) if hasattr(ds, 'dana_all') else 'N/A'}")
print(f"DANA en test (ds.dana_te.sum()):  {int(ds.dana_te.astype(bool).sum())}")
print(f"Total muestras panel: {len(ds.y_all) if hasattr(ds, 'y_all') else 'N/A'}")
print(f"Total muestras test:  {len(ds.y_te)}")
print()
print("Interpretacion:")
print("  P90=86.3x -> pipeline ORIGINAL (86.3mm es el valor original; 89.0mm seria un ETL con datos distintos, ya descartado)")
print("  Si DANA total = 159 -> el checkpoint usa _anomalia() ORIGINAL (full-record)")
print("  Si DANA total = 176 -> el checkpoint usa _anomalia() CORREGIDA (train-only)")

In [ ]:
from scipy import stats
tree_pooled = np.concatenate([rf_aucs, gb_aucs, xgb_aucs])  # 45 valores
gradient_pooled = np.concatenate([lstm_aucs, qnn_aucs, piqnn_aucs, qtl_aucs, lbl_aucs])  # 75 valores
U, p = stats.mannwhitneyu(tree_pooled, gradient_pooled)
print(f"Mann-Whitney U (no pareado): U={U}, p={p}")

In [ ]:
"""
control_3layer_end_to_end.py
==============================================================
Control de profundidad emparejada: LSTM+PI-QNN con 3 capas,
entrenado END-TO-END (sin congelamiento layer-by-layer), para
aislar el efecto de la profundidad del efecto del protocolo LbL.

Comparadores:
  - LSTM+PI-QNN (2 capas, end-to-end)      -- ya tienes estos 15 valores
  - PI-QNN-LbL (3 capas, LbL)               -- ya tienes estos 15 valores
  - LSTM+PI-QNN-3L (3 capas, end-to-end)    -- ESTE experimento, el que falta

Si sigma(3L-end-to-end) ~ sigma(2L-end-to-end) [ambos ~0.017]:
  -> confirma que la reduccion de varianza es del protocolo LbL, no de la profundidad
Si sigma(3L-end-to-end) ~ sigma(PI-QNN-LbL) [ambos ~0.009]:
  -> indicaria que la profundidad por si sola explica la reduccion (raro, pero hay
     que dejar que los datos hablen)
"""

import os
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from modelos import LstmQNN, make_thermodynamic_physics_loss, _evaluar_pytorch

N_Q, N_L = 4, 3          # misma profundidad que PI-QNN-LbL
LAMBDA_CC, LAMBDA_MSE = 0.2, 0.1


def entrenar_3layer_end_to_end(ds, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = LstmQNN(ds.in_dim, n_q=N_Q, n_l=N_L, physics=True).double()

    n_neg = (ds.y_tr == 0).sum().float()
    n_pos = (ds.y_tr == 1).sum().float()
    pos_w = torch.tensor([n_neg / (n_pos + 1e-5)], dtype=torch.float64)
    model.physics_fn = make_thermodynamic_physics_loss(pos_w)

    T_anomaly_tr = torch.tensor([ds.t_anomaly_mes[m] for m in ds.m_tr], dtype=torch.float64)
    dl = DataLoader(TensorDataset(ds.X_tr, ds.y_tr, T_anomaly_tr),
                     batch_size=16, shuffle=True, drop_last=True)

    # Entrenamiento END-TO-END, SIN congelamiento por capas -- todos los
    # parametros entrenables desde la epoca 1, igual que LSTM+PI-QNN (2 capas)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-3, weight_decay=1e-4)
    for ep in range(1, 26):
        model.train()
        for bX, by, bT in dl:
            opt.zero_grad()
            pr, qo = model(bX)
            loss = model.physics_fn(pr, by, qo, bT, ep, 25)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

    met, _ = _evaluar_pytorch(model, ds.X_te, ds.y_te.numpy())
    return met['auc']


if __name__ == '__main__':
    loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
    ds = next((e for e in loaded if hasattr(e, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded

    # Definimos la ruta exacta del checkpoint
    ckpt_path = '/content/control_3layer_e2e_resultados.pt'

    # ---------------------------------------------------------
    # LÓGICA DE REANUDACIÓN (RESUME)
    # ---------------------------------------------------------
    if os.path.exists(ckpt_path):
        # Si el archivo existe, lo cargamos
        aucs_previos = torch.load(ckpt_path, weights_only=False)
        aucs = aucs_previos.tolist() # Convertimos el numpy array a lista de Python
        start_seed = len(aucs)
        print(f"[*] Checkpoint encontrado. Reanudando desde la semilla {start_seed}...")
    else:
        # Si no existe, empezamos limpios
        aucs = []
        start_seed = 0
        print("[*] No se encontró checkpoint. Empezando desde la semilla 0...")

    # El bucle ahora empieza en start_seed en lugar de 0
    for seed in range(start_seed, 15):
        auc = entrenar_3layer_end_to_end(ds, seed)
        aucs.append(auc)
        print(f"Semilla {seed}: AUC={auc:.4f}")
        # Guardamos machacando el archivo anterior con la lista actualizada
        torch.save(np.array(aucs), ckpt_path)

    # Al terminar las 15 semillas, mostramos resultados globales
    aucs = np.array(aucs)
    print("\n" + "="*50)
    print("RESULTADOS GLOBALES: LSTM+PI-QNN-3L (END-TO-END)")
    print("="*50)
    print(f"Mean AUC: {aucs.mean():.4f} ± {aucs.std(ddof=1):.4f}")

    # Descarga automática en Google Colab
    try:
        from google.colab import files
        print("Descargando archivo de resultados...")
        files.download(ckpt_path)
    except ImportError:
        pass

In [ ]:
"""
patch_anomalia_15seed_xgb.py
==============================================================
Aplica el patch de _anomalia() train-only (Finding 1 del critico)
y corre una replica de 15 semillas de XGBoost, comparando contra
el pipeline original (con fuga) para el mismo protocolo de 15
semillas ya usado en el resto del paper.

Tiempo estimado: 5-13 minutos (ver desglose en el chat).
"""

import os
import time
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, recall_score, confusion_matrix
from xgboost import XGBClassifier
from scipy import stats

import etl
import fisica
import features

CSV_PATH = 'datos_limpios_multiestacion.csv'
ANIO_CORTE_TRAIN = 1996

# ============================================================
# 1. Carga de datos (con cache de CSV si existe)
# ============================================================
def cargar_df_diario():
    if os.path.exists(CSV_PATH):
        print(f"[i] Cargando CSV cacheado desde {CSV_PATH}...")
        return pd.read_csv(CSV_PATH, index_col='fecha', parse_dates=['fecha'])
    print("[i] CSV no encontrado, corriendo ETL completo...")
    df = etl.cargar_datos(dir_base='/content/datos')
    df.to_csv(CSV_PATH)
    return df


# ============================================================
# 2. Patch de _anomalia() a train-only (Finding 1 del critico)
# ============================================================
def _anomalia_train_only(agg: pd.DataFrame, col: str) -> pd.Series:
    train_mask = agg.index.year <= ANIO_CORTE_TRAIN
    stats_train = (
        agg[train_mask]
        .groupby(['ciudad', agg[train_mask].index.month])[col]
        .agg(['mean', 'std'])
    )
    stats_train['std'] = stats_train['std'].fillna(0) + 1e-4
    keys = list(zip(agg['ciudad'], agg.index.month))
    means = np.array([stats_train.loc[k, 'mean'] if k in stats_train.index else 0.0 for k in keys])
    stds = np.array([stats_train.loc[k, 'std'] if k in stats_train.index else 1e-4 for k in keys])
    return pd.Series((agg[col].values - means) / stds, index=agg.index)


def construir_features_corregido(df_diario):
    """Pipeline completo con _anomalia() parcheada a train-only."""
    agg = fisica._agregar_mensual(df_diario)
    _anomalia_original = fisica._anomalia
    fisica._anomalia = _anomalia_train_only
    try:
        agg = fisica._indices_sinopticos(agg)
    finally:
        fisica._anomalia = _anomalia_original
    agg = fisica._flag_dana(agg)
    return agg


# ============================================================
# 3. Entrenamiento XGBoost (mismos hiperparametros que el resto del paper)
# ============================================================
def entrenar_evaluar_xgb(ds, seed):
    imp = SimpleImputer(strategy='mean')
    X_tr_clean = imp.fit_transform(ds.X_sk)
    X_te_clean = imp.transform(ds.X_all[ds.s2:])

    n_neg, n_pos = (ds.y_sk == 0).sum(), (ds.y_sk == 1).sum()
    xgb = XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_neg / max(n_pos, 1),
        eval_metric='logloss', random_state=seed, n_jobs=-1
    )
    xgb.fit(X_tr_clean, ds.y_sk, verbose=False)

    y_te = ds.y_te.numpy()
    prob = xgb.predict_proba(X_te_clean)[:, 1]
    pred = (prob >= 0.5).astype(float)

    auc = roc_auc_score(y_te, prob)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

    dana_te = ds.dana_te.astype(bool)
    rc_dana = recall_score(y_te[dana_te], pred[dana_te], zero_division=0) if dana_te.sum() > 0 else np.nan
    rc_p90 = recall_score(ds.y_te_ext.numpy(), pred, zero_division=0)

    return {'auc': auc, 'fpr': fpr, 'rc_dana': rc_dana, 'rc_p90': rc_p90}


# ============================================================
# 4. Ejecucion: 15 semillas, pipeline ORIGINAL vs CORREGIDO
# ============================================================
if __name__ == '__main__':
    t_inicio = time.time()

    df_diario = cargar_df_diario()
    print(f"[i] ETL/carga completada en {(time.time()-t_inicio)/60:.1f} min")

    print("\n[+] Construyendo agg ORIGINAL (con fuga, pipeline actual del paper)...")
    t0 = time.time()
    agg_original = fisica.construir_features(df_diario)
    ds_original = features.preparar_dataset(agg_original)
    print(f"    listo en {time.time()-t0:.1f}s")

    print("\n[+] Construyendo agg CORREGIDO (_anomalia train-only)...")
    t0 = time.time()
    agg_corregido = construir_features_corregido(df_diario)
    ds_corregido = features.preparar_dataset(agg_corregido)
    print(f"    listo en {time.time()-t0:.1f}s")

    resultados = {'original': {'auc': [], 'fpr': [], 'rc_dana': [], 'rc_p90': []},
                  'corregido': {'auc': [], 'fpr': [], 'rc_dana': [], 'rc_p90': []}}

    print("\n[+] Entrenando 15 semillas x 2 pipelines...")
    t0 = time.time()
    for seed in range(15):
        r_orig = entrenar_evaluar_xgb(ds_original, seed)
        r_corr = entrenar_evaluar_xgb(ds_corregido, seed)
        for k in resultados['original']:
            resultados['original'][k].append(r_orig[k])
            resultados['corregido'][k].append(r_corr[k])
        print(f"  [seed {seed}] original AUC={r_orig['auc']:.4f} | corregido AUC={r_corr['auc']:.4f}")

    print(f"\n[i] 15x2 entrenamientos XGBoost completados en {(time.time()-t0)/60:.1f} min")

    print("\n" + "=" * 70)
    print("RESUMEN COMPARATIVO (n=15, mismo protocolo del paper)")
    print("=" * 70)
    for k in ['auc', 'fpr', 'rc_dana', 'rc_p90']:
        orig = np.array(resultados['original'][k])
        corr = np.array(resultados['corregido'][k])
        stat, p = stats.wilcoxon(orig, corr)
        print(f"{k:10s} original: {orig.mean():.4f} ± {orig.std(ddof=1):.4f}  |  "
              f"corregido: {corr.mean():.4f} ± {corr.std(ddof=1):.4f}  |  "
              f"Wilcoxon p={p:.4f}")

    print(f"\nRecuento DANA -- original: {int(agg_original['dana_flag'].sum())} eventos")
    print(f"Recuento DANA -- corregido: {int(agg_corregido['dana_flag'].sum())} eventos")

    import torch
    torch.save(resultados, 'finding1_patch_15seed_resultados.pt')
    print(f"\n[v] Guardado en finding1_patch_15seed_resultados.pt")
    print(f"[v] TIEMPO TOTAL: {(time.time()-t_inicio)/60:.1f} minutos")

In [ ]:
"""
amplitude_embedding_experiment.py
==============================================================
Arquitectura de control: relaja el cuello de botella de 4 dimensiones
mediante Amplitude Embedding (Pérez-Salinas et al. 2020), comprimiendo
a 16 dimensiones (2^4) en vez de 4 antes de entrar al circuito.

Comparador directo: LSTM+QNN (n_layers=2, sin pérdida física),
NO PI-QNN-LbL (que además cambia protocolo de entrenamiento y
profundidad -- comparar contra ese introduciría el mismo confound
de "dos factores a la vez" ya documentado en la Discusión).

Correcciones aplicadas respecto al boceto original:
  1. Añadida no-linealidad (GELU) en la proyección, para evitar el
     mismo patrón de colapso de unidades muertas documentado en el
     control MLP-Match-20 (9/15 semillas colapsando a F1=0).
  2. Modelo completo en float64 (.double()), consistente con el resto
     del pipeline -- se elimina el downcast a float32 de los datos.
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import wilcoxon

from modelos import _loop, _evaluar_pytorch

# ============================================================
# 1. Configuración del dispositivo cuántico
# ============================================================
n_qubits = 4
n_layers = 2  # igual que LSTM+QNN, para comparación directa
dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def amplitude_qnode(inputs, weights):
    # inputs: vector de tamaño 16 por muestra (2^4 = capacidad máxima de amplitud)
    # normalize=False porque la normalización L2 ya se hizo en la capa PyTorch
    qml.AmplitudeEmbedding(features=inputs, wires=range(n_qubits), normalize=False)
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


# ============================================================
# 2. Modelo híbrido
# ============================================================
class LstmAmplitudeQNN(nn.Module):
    """
    Comprime las características a 16 dimensiones (no a 4) y utiliza
    Amplitude Embedding para codificarlas en 4 qubits, ampliando el
    ancho de banda de información que llega al circuito respecto a
    la arquitectura base (LstmQNN, proyección a 4D).
    """
    def __init__(self, in_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(1, 8, 2, batch_first=True, dropout=0.2, bidirectional=True)

        # Proyección a 16 dimensiones (2^4), CON no-linealidad
        self.proj = nn.Sequential(
            nn.Linear(16 + in_dim, 16),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(amplitude_qnode, weight_shapes)

        self.norm = nn.LayerNorm(4)
        self.clf = nn.Linear(4, 1)

    def forward(self, x):
        # x llega en float64, igual que el resto del pipeline -- SIN downcast
        _, (hn, _) = self.lstm(x[:, :12].unsqueeze(-1))
        h = torch.cat([hn[-2], hn[-1]], dim=1)

        st = self.proj(torch.cat([h, x], dim=1))

        # Normalización L2 requerida por Amplitude Embedding (suma de cuadrados = 1)
        # eps por defecto de F.normalize (1e-12) evita división por cero exacta,
        # pero vigila las primeras épocas: si la norma de `st` es casi nula, la
        # salida normalizada es ruido sin señal útil -- exactamente el mismo
        # patrón de colapso ya visto en MLP-Match-20. La GELU añadida arriba
        # mitiga este riesgo respecto al boceto original.
        st_normalized = F.normalize(st, p=2, dim=1)

        q_out = self.qlayer(st_normalized)
        return self.clf(self.norm(q_out)).squeeze(-1)


# ============================================================
# 3. Ejecución de 15 semillas (protocolo CRN, igual que el resto del paper)
# ============================================================
def run_amplitude_experiment(ds, n_seeds=15):
    aucs = []
    print(f"Iniciando evaluación de {n_seeds} semillas para LSTM + Amplitude QNN (16D)...")

    for seed in range(n_seeds):
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = LstmAmplitudeQNN(ds.in_dim).double()
        dl = DataLoader(TensorDataset(ds.X_tr, ds.y_tr), batch_size=16, shuffle=True, drop_last=True)

        model, _ = _loop(model, dl, ds.X_va, ds.y_va, epochs=25, patience=6, lr=5e-3)

        met, _ = _evaluar_pytorch(model, ds.X_te, ds.y_te.numpy())
        aucs.append(met['auc'])
        print(f"  Semilla {seed} -> AUC: {met['auc']:.4f}")

    aucs = np.array(aucs)
    print("\n" + "=" * 50)
    print("RESULTADOS: LSTM + AMPLITUDE QNN (16D)")
    print("=" * 50)
    print(f"AUC medio: {aucs.mean():.4f} ± {aucs.std(ddof=1):.4f} (n={n_seeds})")
    print(f"Max AUC:   {aucs.max():.4f}")

    return aucs


def comparar_con_lstm_qnn(aucs_amplitude, aucs_lstm_qnn):
    """
    Wilcoxon pareado por índice de semilla, justificado por el protocolo
    de números aleatorios comunes (CRN) ya establecido en el paper: la
    semilla i produce el mismo orden de shuffle del DataLoader y la misma
    posición en el stream de RNG de PyTorch para ambas arquitecturas.
    """
    stat, p = wilcoxon(aucs_amplitude, aucs_lstm_qnn)
    print(f"\nWilcoxon (Amplitude-16D vs. LSTM+QNN-4D): W={stat:.2f}, p={p:.4f}")
    print(f"  Amplitude-16D: {np.mean(aucs_amplitude):.4f} ± {np.std(aucs_amplitude, ddof=1):.4f}")
    print(f"  LSTM+QNN-4D:   {np.mean(aucs_lstm_qnn):.4f} ± {np.std(aucs_lstm_qnn, ddof=1):.4f}")
    return stat, p


# ============================================================
# 4. Bloque de ejecución
# ============================================================
if __name__ == '__main__':
    loaded = torch.load('/content/resultados/checkpoints/01_data.pt', map_location='cpu', weights_only=False)
    ds = next((elem for elem in loaded if hasattr(elem, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded

    amplitude_aucs = run_amplitude_experiment(ds)
    torch.save(amplitude_aucs, 'amplitude_16d_resultados.pt')

    # Si ya tienes los 15 valores de LSTM+QNN guardados de la tabla original,
    # cárgalos y descomenta para el test pareado:
    aucs_lstm_qnn = np.array([0.6717, 0.6639, 0.6636, 0.6448, 0.6545, 0.6808,
                               0.6483, 0.6731, 0.6596, 0.6277, 0.6761, 0.6530,
                               0.6369, 0.6544, 0.6750])
    comparar_con_lstm_qnn(amplitude_aucs, aucs_lstm_qnn)

In [ ]:
"""
reuploading_experiment.py
==============================================================
Arquitectura de control: relaja el cuello de botella de información
mediante Data Re-uploading (Perez-Salinas et al. 2020) -- en vez de
comprimir todas las features a una unica proyeccion de 4D antes del
circuito, se codifican 3 subconjuntos DISJUNTOS de 4 features cada
uno, intercalados entre 3 bloques variacionales sucesivos.

NOTA IMPORTANTE DE COMPARABILIDAD (leer antes de interpretar resultados):
Esta arquitectura tiene 3 bloques de profundidad, igual que PI-QNN-LbL,
pero SIN su protocolo de entrenamiento layer-by-layer ni su perdida
fisica, y SIN re-uploading respecto a LSTM+QNN (que tiene 2 capas).
No es directamente comparable 1:1 a ninguna fila existente de la tabla
sin reconocer que aqui cambian simultaneamente DOS factores frente a
LSTM+QNN (profundidad Y re-uploading). Si quieres aislar el efecto del
re-uploading en si mismo, el comparador correcto seria una arquitectura
de 3 capas SIN re-uploading (proyeccion unica a 4D, sin dividir en
subconjuntos), que no esta implementada aqui.

NOTA DE DISEÑO -- embedding gate:
El boceto original usa qml.AngleEmbedding (rotacion en un solo eje,
RX por defecto), distinto del resto del paper (RY+RZ por feature en
LstmQNN). Se mantiene AngleEmbedding por fidelidad a tu diseño original,
pero si prefieres una comparacion mas limpia contra el resto del paper,
sustituye _codificar_bloque() por la version RY+RZ comentada abajo.
"""

import math
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import wilcoxon

from modelos import _loop, _evaluar_pytorch

# ============================================================
# 1. Configuracion del dispositivo cuantico
# ============================================================
n_qubits = 4
n_bloques = 3          # 3 subconjuntos de 4 features = 12 features totales
n_features_per_block = 4
dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def reuploading_qnode(inputs, weights):
    """
    inputs: vector de 12 dimensiones (3 bloques de 4 features cada uno)
    weights: forma (3, n_qubits, 3) -- un StronglyEntanglingLayers por bloque
    """
    for b in range(n_bloques):
        start = b * n_features_per_block
        end = start + n_features_per_block
        # qml.AngleEmbedding(inputs[..., start:end], wires=range(n_qubits))
        # --- Alternativa RY+RZ, consistente con el resto del paper: ---
        for i in range(n_qubits):
            qml.RY(inputs[..., start + i], wires=i)
            qml.RZ(inputs[..., start + i], wires=i)
        qml.StronglyEntanglingLayers(weights[b:b + 1], wires=range(n_qubits))

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


# ============================================================
# 2. Modelo hibrido
# ============================================================
class LstmReuploadingQNN(nn.Module):
    """
    Proyecta a 12 dimensiones (3 x 4) y las re-inyecta en el circuito
    en 3 bloques disjuntos intercalados con capas variacionales,
    ampliando el ancho de banda de informacion sin aumentar qubits.
    """
    def __init__(self, in_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(1, 8, 2, batch_first=True, dropout=0.2, bidirectional=True)

        # Proyeccion a 12 dimensiones, CON no-linealidad
        self.proj = nn.Sequential(
            nn.Linear(16 + in_dim, n_bloques * n_features_per_block),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        weight_shapes = {"weights": (n_bloques, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(reuploading_qnode, weight_shapes)

        self.norm = nn.LayerNorm(4)
        self.clf = nn.Linear(4, 1)

    def forward(self, x):
        # x en float64, consistente con el resto del pipeline
        _, (hn, _) = self.lstm(x[:, :12].unsqueeze(-1))
        h = torch.cat([hn[-2], hn[-1]], dim=1)

        st = self.proj(torch.cat([h, x], dim=1))

        # Los angulos de AngleEmbedding no requieren normalizacion L2
        # (a diferencia de AmplitudeEmbedding), pero conviene acotar el
        # rango para estabilidad numerica del circuito, igual que en
        # LstmQNN original (sigmoid * pi):
        st_scaled = torch.sigmoid(st) * math.pi

        q_out = self.qlayer(st_scaled)
        return self.clf(self.norm(q_out)).squeeze(-1)


# ============================================================
# 3. Ejecucion de 15 semillas (protocolo CRN)
# ============================================================
def run_reuploading_experiment(ds, n_seeds=15):
    aucs = []
    print(f"Iniciando evaluacion de {n_seeds} semillas para LSTM + Re-uploading QNN (3x4D)...")

    for seed in range(n_seeds):
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = LstmReuploadingQNN(ds.in_dim).double()
        dl = DataLoader(TensorDataset(ds.X_tr, ds.y_tr), batch_size=16, shuffle=True, drop_last=True)

        model, _ = _loop(model, dl, ds.X_va, ds.y_va, epochs=25, patience=6, lr=5e-3)

        met, _ = _evaluar_pytorch(model, ds.X_te, ds.y_te.numpy())
        aucs.append(met['auc'])
        print(f"  Semilla {seed} -> AUC: {met['auc']:.4f}")

    aucs = np.array(aucs)
    print("\n" + "=" * 50)
    print("RESULTADOS: LSTM + RE-UPLOADING QNN (3x4D)")
    print("=" * 50)
    print(f"AUC medio: {aucs.mean():.4f} +/- {aucs.std(ddof=1):.4f} (n={n_seeds})")
    print(f"Max AUC:   {aucs.max():.4f}")

    return aucs


def comparar_con_baselines(aucs_reuploading, aucs_lstm_qnn=None, aucs_lbl=None):
    """
    Comparaciones pareadas por indice de semilla (protocolo CRN).
    Se comparan ambos baselines por separado dado que ninguno controla
    exactamente los mismos factores (ver nota de comparabilidad arriba).
    """
    if aucs_lstm_qnn is not None:
        stat, p = wilcoxon(aucs_reuploading, aucs_lstm_qnn)
        print(f"\nWilcoxon (Reuploading vs. LSTM+QNN, 2 capas, sin re-upload): W={stat:.2f}, p={p:.4f}")
        print("  NOTA: esta comparacion mezcla el efecto de re-uploading con el de profundidad (3 vs 2).")

    if aucs_lbl is not None:
        stat2, p2 = wilcoxon(aucs_reuploading, aucs_lbl)
        print(f"\nWilcoxon (Reuploading vs. PI-QNN-LbL, misma profundidad=3): W={stat2:.2f}, p={p2:.4f}")
        print("  NOTA: esta comparacion mezcla re-uploading con protocolo LbL + perdida fisica.")


# ============================================================
# 4. Bloque de ejecucion
# ============================================================
if __name__ == '__main__':
    loaded = torch.load('/content/resultados/checkpoints/01_data.pt', map_location='cpu', weights_only=False)
    ds = next((elem for elem in loaded if hasattr(elem, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded

    reuploading_aucs = run_reuploading_experiment(ds)
    torch.save(reuploading_aucs, 'reuploading_resultados.pt')

    aucs_lstm_qnn = np.array([0.6717, 0.6639, 0.6636, 0.6448, 0.6545, 0.6808,
                               0.6483, 0.6731, 0.6596, 0.6277, 0.6761, 0.6530,
                               0.6369, 0.6544, 0.6750])
    aucs_lbl = np.array([0.6487, 0.6600, 0.6644, 0.6538, 0.6454, 0.6655,
                          0.6812, 0.6707, 0.6659, 0.6597, 0.6583, 0.6518,
                          0.6618, 0.6586, 0.6621])
    comparar_con_baselines(reuploading_aucs, aucs_lstm_qnn, aucs_lbl)

In [ ]:
"""
ADAPTACIÓN PARA GOOGLE COLAB (VERSIÓN DEFINITIVA TERMODINÁMICA - CC + MSE)
Orquestador del pipeline PI-QNN Alicante con QTL, LbL, QKSVM, SARIMAX y DeLong/Bootstrap
-- INCLUYE SISTEMA DE CHECKPOINTS ANTI-CRASH --
"""

import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import torch

# ── 1. Adaptación de Rutas para Colab ─────────────────────────────────────────
sys.path.append('/content')

warnings.filterwarnings('ignore')
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
np.random.seed(42)

from etl           import cargar_datos
from fisica        import construir_features
from features      import preparar_dataset
from validacion    import ejecutar_validacion
from visualizacion import generar_figuras
from experimento_ruido import ejecutar_evaluacion_ruido

from modelos       import (entrenar_baselines, entrenar_qml,
                           entrenar_qksvm, entrenar_ablacion, analisis_extremos,
                           entrenar_qtl, entrenar_qnn_layerbylayer,
                           entrenar_baselines_multisemilla)

# ── 2. Adaptación de Argumentos ───────────────────────────────────────────────
class Args:
    # IMPORTANTE: Asegúrate de que en Colab subes los txt dentro de /content/Barcelona, /content/Valencia...
    datos = '/content/datos'
    salida = '/content/resultados'
    skip_qml = False
    #skip_qml = True

    # NUEVO: Lista de ciudades multi-estación
    ciudades = ['Barcelona', 'Valencia', 'Alicante', 'San_Javier']

    # Control de Checkpoints
    usar_checkpoints = True    # Si es True, carga desde disco si existe
    forzar_reentreno = False   # Ponlo a True si cambiaste el código y quieres reescribir TODOS los checkpoints

    # 🎯 FIX: flag independiente para invalidar solo el checkpoint QML (03_qml.pt)
    # sin tener que forzar el reentreno de baselines/QKSVM/ablación también.
    # Actívalo la próxima vez que ejecutes tras cambios en modelos.py que afecten
    # a entrenar_qml/entrenar_qtl/entrenar_qnn_layerbylayer (p.ej. el fix del
    # pos_weight de QTL), o el checkpoint seguirá sirviendo resultados obsoletos.
    forzar_reentreno_qml = False

args = Args()

# ══════════════════════════════════════════════════════════════════════════════
def main():
    os.makedirs(args.salida, exist_ok=True)
    ckpt_dir = os.path.join(args.salida, 'checkpoints')
    os.makedirs(ckpt_dir, exist_ok=True)

    # ── RUTAS DE CHECKPOINTS ──
    ckpt_data  = f"{ckpt_dir}/01_data.pt"
    ckpt_base  = f"{ckpt_dir}/02_baselines.pt"
    ckpt_qml   = f"{ckpt_dir}/03_qml.pt"
    ckpt_qksvm = f"{ckpt_dir}/04_qksvm.pt"
    ckpt_abl   = f"{ckpt_dir}/05_ablacion.pt"
    ckpt_ruido_pesos = f"{ckpt_dir}/06_pesos_ruido.pt"
    ckpt_ruido_metricas = f"{ckpt_dir}/07_metricas_ruido.pt"
    ckpt_val   = f"{ckpt_dir}/08_validacion.pt"
    ckpt_multisemilla = f"{ckpt_dir}/09_multisemilla.pt"
    ckpt_multisemilla_clasico = f"{ckpt_dir}/10_multisemilla_clasico.pt"

    # 🎯 FIX: combinación efectiva de "forzar reentreno" para la etapa QML
    forzar_qml = args.forzar_reentreno or args.forzar_reentreno_qml

    # =====================================================================
    # PASOS 1, 2 y 3: ETL y FÍSICA MULTI-ESTACIÓN
    # =====================================================================
    print("\n" + "═"*65)
    print("PASOS 1-3 · ETL Multi-Estación, Física Diagnóstica y Features")
    print("═"*65)

    if args.usar_checkpoints and os.path.exists(ckpt_data) and not args.forzar_reentreno:
        print("  [+] Cargando datos procesados desde checkpoint...")
        agg, ds = torch.load(ckpt_data, weights_only=False)
    else:
        # 1. Cargamos las 4 ciudades (devuelve el df con la columna string 'ciudad')
        df_diario = cargar_datos(args.datos, ciudades=args.ciudades)

        # 2. Construir features físicas
        # (fisica.py debe usar la columna 'ciudad' para buscar la latitud de cada fila)
        agg = construir_features(df_diario)

        # 3. ONE-HOT ENCODING (CRÍTICO PARA QML Y XGBOOST)
        # Transformamos la columna texto 'ciudad' en 4 columnas (0.0 o 1.0)
        # ej: 'ciudad_Barcelona', 'ciudad_Valencia'...
        if 'ciudad' in agg.columns:
            agg = pd.get_dummies(agg, columns=['ciudad'], dtype=float)

        # 4. Preparar tensores (Split Train/Val/Test)
        ds = preparar_dataset(agg)

        torch.save((agg, ds), ckpt_data)
        print("  [v] Datos multi-estación procesados, codificados y guardados en checkpoint.")

    # =====================================================================
    # PASO 4: BASELINES CLÁSICOS Y RÉPLICA MULTI-SEMILLA
    # =====================================================================
    print("\n" + "═"*65)
    print("PASO 4 · Baselines Clásicos y Estadísticos")
    print("═"*65)

    if args.usar_checkpoints and os.path.exists(ckpt_base) and not args.forzar_reentreno:
        print("  [+] Cargando modelos clásicos (single-run) desde checkpoint...")
        res_base = torch.load(ckpt_base, weights_only=False)
    else:
        res_base = entrenar_baselines(ds)
        torch.save(res_base, ckpt_base)
        print("  [v] Baselines (single-run) entrenados y guardados en checkpoint.")

    # ── Réplica Multi-Semilla Clásica (15 semillas) ──
    if args.usar_checkpoints and os.path.exists(ckpt_multisemilla_clasico) and not args.forzar_reentreno:
        print("  [+] Cargando réplica multi-semilla clásica desde checkpoint...")
        resultados_cls_multi = torch.load(ckpt_multisemilla_clasico, weights_only=False)
    else:
        print("  [!] Ejecutando réplica Multi-Semilla Clásica (15 semillas)...")
        resultados_cls_multi = entrenar_baselines_multisemilla(ds, seeds=range(15))
        torch.save(resultados_cls_multi, ckpt_multisemilla_clasico)
        print("  [v] Multi-semilla clásica completada y guardada en checkpoint.")

    # Imprimir la tabla de resumen
    print("\n" + "="*80)
    print(f" {'Modelo Clásico':<20} | {'AUC':<15} | {'FPR':<15} | {'Rc-DANA':<15} ")
    print("-" * 80)

    for nm, met in resultados_cls_multi.items():
        auc_m, auc_s = met['auc']['mean'], met['auc']['std']
        fpr_m, fpr_s = met['fpr']['mean'], met['fpr']['std']
        rcd_m, rcd_s = met['rc_dana']['mean'], met['rc_dana']['std']

        print(f" {nm:<20} | {auc_m:.3f} ± {auc_s:.3f} | {fpr_m:.3f} ± {fpr_s:.3f} | {rcd_m:.3f} ± {rcd_s:.3f} ")
    print("="*80)

    # =====================================================================
    # PASO 5: MODELOS QML (Base, QTL, LbL)
    # =====================================================================
    if args.skip_qml:
        print("\n  [--skip-qml] Saltando pasos cuánticos")
        res_qml, res_qksvm, res_abl, res_val, res_ruido = {'_probs': {}}, {}, {}, {}, {}
    else:
        print("\n" + "═"*65)
        print("PASO 5 · Modelos QML (Base, PI-QNN Termodinámica, QTL y LbL)")
        print("═"*65)

        if args.usar_checkpoints and os.path.exists(ckpt_qml) and not forzar_qml:
            print("  [+] Cargando modelos cuánticos (QML) desde checkpoint...")
            res_qml = torch.load(ckpt_qml, weights_only=False)
        else:
            res_qml = entrenar_qml(ds)

            res_qml['_probs'] = {}
            for nm in ['LSTM+QNN', 'LSTM+PI-QNN']:
                if nm in res_qml and '_prob' in res_qml[nm]:
                    res_qml['_probs'][nm] = res_qml[nm]['_prob']

            if '_modelo_lstm' in res_base:
                res_qtl = entrenar_qtl(ds, res_base['_modelo_lstm'])
                res_qml.update(res_qtl)
                if '_probs' not in res_qml: res_qml['_probs'] = {}
                res_qml['_probs']['LSTM+QTL'] = res_qtl['LSTM+QTL']['_prob']
            else:
                print("  [!] No se pudo entrenar QTL porque falta '_modelo_lstm'")

            res_lbl = entrenar_qnn_layerbylayer(ds)
            res_qml.update(res_lbl)
            res_qml['_probs']['PI-QNN-LbL'] = res_lbl['PI-QNN-LbL']['_prob']

            # 🎯 1. Guardamos LOS PESOS para el ruido ANTES de borrarlos
            if 'LSTM+QNN' in res_qml and 'PI-QNN-LbL' in res_qml:
                torch.save({
                    'LSTM+QNN_state': res_qml['LSTM+QNN']['_modelo'].state_dict(),
                    'PI-QNN-LbL_state': res_qml['PI-QNN-LbL']['_modelo'].state_dict(),
                }, ckpt_ruido_pesos)

            # 🎯 2. ANTI-CRASH: Limpiamos los modelos de la memoria
            for nm in ['LSTM+QNN', 'LSTM+PI-QNN', 'LSTM+QTL', 'PI-QNN-LbL']:
                if nm in res_qml and '_modelo' in res_qml[nm]:
                    del res_qml[nm]['_modelo']

            # 🎯 3. Guardamos el checkpoint QML limpio (sin QNodes que rompan pickle)
            torch.save(res_qml, ckpt_qml)
            print("  [v] Modelos QML entrenados y guardados en checkpoint.")

        # =====================================================================
        # PASO 5.5: SIMULACIÓN DE RUIDO NISQ
        # =====================================================================
        print("\n" + "═"*65)
        print("PASO EXTRA · Simulación de Hardware NISQ (Ruido)")
        print("═"*65)

        # 🎯 FIX: si el checkpoint QML se acaba de regenerar (forzar_qml=True),
        # las métricas de ruido cacheadas quedan desactualizadas también,
        # porque dependen de los pesos que acabamos de recalcular.
        if args.usar_checkpoints and os.path.exists(ckpt_ruido_metricas) and not args.forzar_reentreno and not forzar_qml:
            print("  [+] Cargando métricas de ruido cuántico desde checkpoint...")
            res_ruido = torch.load(ckpt_ruido_metricas, weights_only=False)
        else:
            # Verificamos si tenemos el archivo de pesos guardado en disco
            if os.path.exists(ckpt_ruido_pesos):
                print("  [+] Reconstruyendo modelos desde los pesos para la simulación...")
                from modelos import LstmQNN

                pesos = torch.load(ckpt_ruido_pesos, weights_only=False)

                # Reconstruimos la arquitectura EXACTA del modelo base (2 capas, sin física)
                # 🎯 FIX: physics=False/True explícitos en vez de None/'cc_mse'.
                # LstmQNN.physics se usa como booleano puro en forward() (bool(self.physics)),
                # así que None/'cc_mse' "funcionaban" por la truthiness de Python, pero es
                # una trampa si algún día el chequeo cambia a `if self.physics is True`.
                modelo_qnn = LstmQNN(ds.in_dim, n_q=4, n_l=2, physics=False).double()
                modelo_qnn.load_state_dict(pesos['LSTM+QNN_state'])

                # Reconstruimos la arquitectura EXACTA del Layer-by-Layer (3 capas, CC+MSE)
                modelo_lbl = LstmQNN(ds.in_dim, n_q=4, n_l=3, physics=True).double()
                modelo_lbl.load_state_dict(pesos['PI-QNN-LbL_state'])

                print("  [!] Ejecutando simulación de hardware NISQ (Ruido p=0.01)...")
                res_ruido = ejecutar_evaluacion_ruido(
                    ds, modelo_qnn, modelo_lbl, p_error=0.01
                )
                torch.save(res_ruido, ckpt_ruido_metricas)
                print("  [v] Métricas de ruido completadas y guardadas.")
            else:
                print("  [!] ERROR: No se encontró el archivo de pesos '06_pesos_ruido.pt'.")
                print("      Debes apagar args.usar_checkpoints una vez para que se generen.")
                res_ruido = {}

        # =====================================================================
        # PASO 6: QKSVM
        # =====================================================================
        print("\n" + "═"*65)
        print("PASO 6 · Quantum Kernel Support Vector Machine (QKSVM)")
        print("═"*65)

        if args.usar_checkpoints and os.path.exists(ckpt_qksvm) and not args.forzar_reentreno:
            print("  [+] Cargando QKSVM desde checkpoint...")
            res_qksvm = torch.load(ckpt_qksvm, weights_only=False)
        else:
            res_qksvm = entrenar_qksvm(ds)
            torch.save(res_qksvm, ckpt_qksvm)
            print("  [v] QKSVM entrenado y guardado en checkpoint.")

        # =====================================================================
        # PASO 7: ABLACIÓN
        # =====================================================================
        print("\n" + "═"*65)
        print("PASO 7 · Ablación cuántica")
        print("═"*65)

        res_abl_cache = None
        if args.usar_checkpoints and os.path.exists(ckpt_abl) and not args.forzar_reentreno:
            res_abl_cache = torch.load(ckpt_abl, weights_only=False)
            if not res_abl_cache:  # 🎯 protege contra checkpoints vacíos guardados por error
                print("  [!] Checkpoint de ablación vacío detectado — se descarta y se reentrena.")
                res_abl_cache = None

        if res_abl_cache is not None:
            print("  [+] Cargando Ablación desde checkpoint...")
            res_abl = res_abl_cache
        else:
            res_abl = entrenar_ablacion(ds)
            torch.save(res_abl, ckpt_abl)
            print("  [v] Ablación entrenada y guardada en checkpoint.")

    # =====================================================================
    # PASOS FINALES: ANÁLISIS, JSON Y FIGURAS
    # =====================================================================
    print("\n" + "═"*65)
    print("PASO 8 · Análisis de eventos extremos (DANA / p90)")
    print("═"*65)
    res_extremos = analisis_extremos(ds, res_base, res_qml, res_qksvm if not args.skip_qml else None)

    print("\n" + "═"*65)
    print("PASO 9 · Validación estadística (Bootstrapping & DeLong)")
    print("═"*65)
    if args.skip_qml:
        res_val = {}
    elif args.usar_checkpoints and os.path.exists(ckpt_val) and not args.forzar_reentreno and not forzar_qml:
        print("  [+] Cargando resultados de Validación y Nyström desde checkpoint...")
        res_val = torch.load(ckpt_val, weights_only=False)
    else:
        res_val = ejecutar_validacion(ds, res_base, res_qml, res_qksvm, skip_nystrom=False)
        torch.save(res_val, ckpt_val)
        print("  [v] Validación y Nyström completados y guardados en checkpoint.")

    # PASO 10 — GUARDAR MÉTRICAS JSON (Limpieza)
    def _clean(d):
        if d is None: return None
        if not isinstance(d, dict):
            if isinstance(d, (float, np.floating)): return float(d)
            if isinstance(d, (int, np.integer)): return int(d)
            if isinstance(d, (bool, np.bool_)): return bool(d)
            if isinstance(d, (list, tuple)): return [_clean(i) for i in d]
            return d
        return {str(k): _clean(v) for k, v in d.items() if not str(k).startswith('_')}

    metricas_json = {
        "baselines":  _clean(res_base),
        "qml":        _clean(res_qml),
        "qksvm":      _clean(res_qksvm if not args.skip_qml else {}),
        "ablacion":   _clean(res_abl if not args.skip_qml else {}),
        "extremos":   _clean(res_extremos),
        "ruido":      _clean(res_ruido),
        "validacion": _clean(res_val),
        "fisica": {
            "umbral_p60":   float(ds.umbral),
            "umbral_p90":   float(ds.umbral_ext),
            "n_dana_test":  int(ds.dana_te.sum()),
            "in_dim":       ds.in_dim,
            "termodinamica_aplicada": True
        },
    }
    ruta_json = f"{args.salida}/main_metrics.json"
    with open(ruta_json, 'w') as f:
        json.dump(metricas_json, f, indent=2)

    # PASO 11 — FIGURAS DEL PAPER
    print("\n" + "═"*65)
    print("PASO 11 · Generando figuras y PDF")
    print("═"*65)

    res_qml_figs = res_qml.copy()
    if not args.skip_qml and res_qksvm:
        res_qml_figs.update(res_qksvm)
        if '_probs' not in res_qml_figs: res_qml_figs['_probs'] = {}
        res_qml_figs['_probs']['QKSVM'] = res_qksvm['QKSVM']['_prob']

    # Extract the dynamic baseline from the current run (fallback to hardcoded if missing)
    lbl_res = res_qml.get('PI-QNN-LbL', {})
    if not lbl_res:
        print("  [!] AVISO: 'PI-QNN-LbL' no está en res_qml — las figuras usarán "
              "el baseline hardcodeado (acc=61.36%, F1=0.555), NO los resultados "
              "de esta corrida. Revisa por qué falta antes de dar el PDF por bueno.")

    # 🎯 NUEVO: Cargar el diccionario multisemilla antes de generar las figuras
    res_multi = {}
    if os.path.exists(ckpt_multisemilla):
        print("  [+] Cargando datos multi-semilla para el gráfico de violines...")
        res_multi = torch.load(ckpt_multisemilla, weights_only=False)

    # Llamada actualizada a la función
    generar_figuras(
        agg,
        ds,
        res_base,
        res_qml_figs,
        res_abl if not args.skip_qml else {},
        None,
        None,
        res_validacion=res_val,
        salida=args.salida,
        baseline_acc=lbl_res.get('acc', 0.6136) * 100,
        baseline_f1=lbl_res.get('f1', 0.555),
        res_multisemilla=res_multi  # <-- 🎯 AÑADIDO AQUÍ
    )

    _resumen(res_base, res_qml, res_qksvm if not args.skip_qml else {}, res_val, metricas_json, args.salida)

# ── Resumen de consola ────────────────────────────────────────────────────────
def _resumen(res_base, res_qml, res_qksvm, res_val, metricas_json, salida):
    print("\n" + "═"*65)
    print("RESUMEN FINAL")
    print("═"*65)

    todos = {k: v for k, v in res_base.items()  if not k.startswith('_')}
    todos.update({k: v for k, v in res_qml.items() if not k.startswith('_')})
    if res_qksvm:
        todos.update({k: v for k, v in res_qksvm.items() if not k.startswith('_')})

    print(f"\n  {'Modelo':<22} {'Accuracy':>9}  {'F1':>6}  {'AUC':>6}")
    print("  " + "-"*48)
    for nm, mv in todos.items():
        auc_s = f"{mv['auc']:.3f}" if mv.get('auc') else "  —  "
        mk    = " ◄ PI-QNN" if "PI" in nm else (" ◄ QTL" if "QTL" in nm else (" ◄ QKSVM" if "QKSVM" in nm else (" ◄ QNN" if "QNN" in nm else "")))
        print(f"  {nm:<22} {mv['acc']*100:>8.2f}%  {mv['f1']:>6.3f}  {auc_s:>6}{mk}")

    tests = res_val.get('tests_delong', {})
    if tests:
        print(f"\n  Tests de DeLong (significancia AUC):")
        for etiq, t in tests.items():
            sig = "✅ sig." if t['significativo_95'] else "⚠️  no sig."
            nm_a = etiq.split('_vs_')[0].replace('_', '+')
            nm_b = etiq.split('_vs_')[1].replace('_', ' ')
            print(f"    {nm_a:<18} vs {nm_b:<18}  ΔAUC={t['delta_auc']:+.4f}  p={t['p_valor']:.4f}  {sig}")

    # 🎯 NUEVO: Imprimir los resultados del Ruido Cuántico (Formato Multi-seed Monte Carlo)
    res_ruido = metricas_json.get('ruido', {})
    if res_ruido:
        print(f"\n  [NISQ] Impacto del Ruido Cuántico (Monte Carlo 15 seeds, p=0.01):")
        for nm, mv in res_ruido.items():
            acc_s = mv.get('acc', 0) * 100

            # Extraemos medias y desviaciones (si no existen, ponemos 0)
            f1_mean, f1_std = mv.get('f1', 0), mv.get('f1_std', 0)
            rc_dana_mean, rc_dana_std = mv.get('rc_dana', 0), mv.get('rc_dana_std', 0)
            fpr_mean, fpr_std = mv.get('fpr', 0), mv.get('fpr_std', 0)

            print(f"    {nm:<18} Acc: {acc_s:>5.2f}% | F1: {f1_mean:.3f}±{f1_std:.3f} | Rc-DANA: {rc_dana_mean:.3f}±{rc_dana_std:.3f} | FPR: {fpr_mean:.3f}±{fpr_std:.3f}")

    print(f"\n  Archivos generados en {salida}/:")
    for f in ['main_metrics.json', 'paper_figures.pdf', 'tabla_latex.tex', 'checkpoints/']:
        print(f"    ✅  {f}")

    print("\n" + "═"*65)
    print("✅  PIPELINE COMPLETO")
    print("═"*65)

# ── Ejecución de la celda en Colab ───────────────────────────────────────────
main()

In [ ]:
"""
verificacion_anomalia_leakage.py (v2)
==========================================
Requiere: datos_limpios_multiestacion.csv (generado por etl.cargar_datos())
Si no lo tienes, ejecuta primero etl.cargar_datos() una vez para generarlo,
o ajusta CSV_PATH / la ruta de los .txt crudos.
"""

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

import etl
import fisica
import features


def cargar_df_diario():
    return etl.cargar_datos(dir_base='/content/datos')

df_diario = cargar_df_diario()
df_diario.to_csv('/content/datos_limpios_multiestacion.csv')  # cache para la próxima vez


ANIO_CORTE_TRAIN = 1996

def _anomalia_train_only(agg: pd.DataFrame, col: str) -> pd.Series:
    train_mask = agg.index.year <= ANIO_CORTE_TRAIN
    stats = (
        agg[train_mask]
        .groupby(['ciudad', agg[train_mask].index.month])[col]
        .agg(['mean', 'std'])
    )
    stats['std'] = stats['std'].fillna(0) + 1e-4
    keys = list(zip(agg['ciudad'], agg.index.month))
    means = np.array([stats.loc[k, 'mean'] if k in stats.index else 0.0 for k in keys])
    stds = np.array([stats.loc[k, 'std'] if k in stats.index else 1e-4 for k in keys])
    return pd.Series((agg[col].values - means) / stds, index=agg.index)


def construir_features_train_only(df_diario):
    agg = fisica._agregar_mensual(df_diario)
    _anomalia_original = fisica._anomalia
    fisica._anomalia = _anomalia_train_only
    try:
        agg = fisica._indices_sinopticos(agg)
    finally:
        fisica._anomalia = _anomalia_original
    agg = fisica._flag_dana(agg)
    return agg


def entrenar_y_evaluar_xgb(ds, seed=42):
    imp = SimpleImputer(strategy='mean')
    X_tr_clean = imp.fit_transform(ds.X_sk)
    X_te_clean = imp.transform(ds.X_all[ds.s2:])
    n_neg, n_pos = (ds.y_sk == 0).sum(), (ds.y_sk == 1).sum()
    xgb = XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_neg / max(n_pos, 1),
        eval_metric='logloss', random_state=seed, n_jobs=-1
    )
    xgb.fit(X_tr_clean, ds.y_sk, verbose=False)
    prob = xgb.predict_proba(X_te_clean)[:, 1]
    return roc_auc_score(ds.y_te.numpy(), prob)


if __name__ == '__main__':
    df_diario = cargar_df_diario()

    print("[2/4] Construyendo features -- VERSIÓN ORIGINAL (con fuga parcial)...")
    agg_original = fisica.construir_features(df_diario)
    ds_original = features.preparar_dataset(agg_original)
    auc_original = entrenar_y_evaluar_xgb(ds_original)

    print("[3/4] Construyendo features -- VERSIÓN TRAIN-ONLY (sin fuga)...")
    agg_train_only = construir_features_train_only(df_diario)
    ds_train_only = features.preparar_dataset(agg_train_only)
    auc_train_only = entrenar_y_evaluar_xgb(ds_train_only)

    print("\n" + "=" * 60)
    print("RESULTADOS")
    print("=" * 60)
    print(f"AUC con _anomalia() full-record (original): {auc_original:.4f}")
    print(f"AUC con _anomalia() train-only (<=1996):     {auc_train_only:.4f}")
    print(f"Diferencia:                                   {auc_train_only - auc_original:+.4f}")

    print("\nMagnitud del cambio en las variables de anomalía (media |delta| en todo el panel):")
    for col in ['iap', 'iav', 'ia_racha', 'ia_sol', 'ia_dp_std']:
        delta = (agg_train_only[col] - agg_original[col]).abs().mean()
        print(f"  {col:12s}: |Δ| medio = {delta:.4f}")

In [ ]:
import torch
loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
ds_checkpoint = next((e for e in loaded if hasattr(e, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded
print(f"umbral (P60) en el checkpoint: {ds_checkpoint.umbral}")
print(f"umbral_ext (P90) en el checkpoint: {ds_checkpoint.umbral_ext}")

In [ ]:
from pathlib import Path
import pandas as pd

def resumen_archivo(ruta, col_indices):
    """Cuenta líneas válidas y rango de fechas de un .txt AEMET, sin reconstruir el pipeline."""
    from etl import _leer_aemet
    registros = _leer_aemet(ruta, col_indices)
    if not registros:
        return {'n_filas': 0, 'fecha_min': None, 'fecha_max': None}
    fechas = sorted(registros.keys())
    return {'n_filas': len(registros), 'fecha_min': fechas[0], 'fecha_max': fechas[-1]}

base = Path('/content/datos')
ciudades = ['Barcelona', 'Valencia', 'Alicante', 'San_Javier']
archivos = {
    'precipitacion': [1], 'presion': [1, 2], 'temperatura': [1, 2],
    'sol': [1, 2], 'viento': [1, 3],
}

for ciudad in ciudades:
    print(f"\n=== {ciudad} ===")
    for nombre, cols in archivos.items():
        ruta = base / ciudad / f'datos_{nombre}.txt'
        info = resumen_archivo(ruta, cols)
        print(f"  {nombre:15s}: {info['n_filas']:>7,} filas | "
              f"{info['fecha_min']} → {info['fecha_max']}")

In [ ]:
import torch
loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
ds = next((e for e in loaded if hasattr(e, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded

# Necesitamos saber si hay muestras de San_Javier en el periodo de test (s2:)
import numpy as np
city_cols = [c for c in ds.feature_names if c.startswith('ciudad_')]
print("Columnas de ciudad:", city_cols)

# Índice de la columna ciudad_San_Javier dentro de X_all
idx_sj = ds.feature_names.index('ciudad_San_Javier')
es_sj_test = ds.X_all[ds.s2:, idx_sj] == 1.0
print(f"Muestras de San Javier en el test set: {es_sj_test.sum()} de {len(es_sj_test)} totales")

# Si quieres ver la distribución temporal, usa meses_all para ese tramo
meses_sj_test = ds.meses_all[ds.s2:][es_sj_test]
print(f"Meses de San Javier en test (muestra): {meses_sj_test[:20]}")

In [ ]:
# ============================================================
# ablacion_ia_racha.py
# Diagnóstico: ¿depende Rc-DANA de tener ia_racha como input,
# dado que ia_racha es la mitad de la definición de dana_flag?
# Requiere: 01_data.pt (o el checkpoint donde tengas ds con
# X_sk, y_sk, X_all, y_te, dana_te, y_te_ext, feature_names)
# ============================================================
import torch
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, recall_score, confusion_matrix
from xgboost import XGBClassifier
from scipy import stats

loaded = torch.load("/content/resultados/checkpoints/01_data.pt", weights_only=False)
ds = next((e for e in loaded if hasattr(e, 'in_dim')), loaded[-1]) if isinstance(loaded, (tuple, list)) else loaded

feature_names = list(ds.feature_names)
mask_sin_ia_racha = np.array([nm != 'ia_racha' for nm in feature_names])

y_te = ds.y_te.numpy()
y_te_ext = ds.y_te_ext.numpy()
dana_te = ds.dana_te.astype(bool) if not hasattr(ds.dana_te, 'numpy') else ds.dana_te.numpy().astype(bool)


def entrenar_y_evaluar(X_sk, X_te, seed):
    imp = SimpleImputer(strategy='mean')
    X_tr_clean = imp.fit_transform(X_sk)
    X_te_clean = imp.transform(X_te)

    n_neg, n_pos = (ds.y_sk == 0).sum(), (ds.y_sk == 1).sum()
    xgb = XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_neg / max(n_pos, 1),
        eval_metric='logloss', random_state=seed, n_jobs=-1
    )
    xgb.fit(X_tr_clean, ds.y_sk, verbose=False)

    prob = xgb.predict_proba(X_te_clean)[:, 1]
    pred = (prob >= 0.5).astype(float)

    auc = roc_auc_score(y_te, prob)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    rc_dana = recall_score(y_te[dana_te], pred[dana_te], zero_division=0) if dana_te.sum() > 0 else np.nan
    rc_p90 = recall_score(y_te_ext, pred, zero_division=0)

    return {'auc': auc, 'fpr': fpr, 'rc_dana': rc_dana, 'rc_p90': rc_p90}


N_SEEDS = 15
X_te_full = ds.X_all[ds.s2:]
X_te_sin = X_te_full[:, mask_sin_ia_racha]
X_sk_sin = ds.X_sk[:, mask_sin_ia_racha]

resultados_con = {k: [] for k in ['auc', 'fpr', 'rc_dana', 'rc_p90']}
resultados_sin = {k: [] for k in ['auc', 'fpr', 'rc_dana', 'rc_p90']}

for seed in range(N_SEEDS):
    r_con = entrenar_y_evaluar(ds.X_sk, X_te_full, seed)
    r_sin = entrenar_y_evaluar(X_sk_sin, X_te_sin, seed)
    for k in resultados_con:
        resultados_con[k].append(r_con[k])
        resultados_sin[k].append(r_sin[k])
    print(f"[seed {seed}] CON: AUC={r_con['auc']:.4f} Rc-DANA={r_con['rc_dana']:.4f}  |  "
          f"SIN: AUC={r_sin['auc']:.4f} Rc-DANA={r_sin['rc_dana']:.4f}")

print("\n" + "=" * 70)
print("RESUMEN (mean ± std, n=15)")
print("=" * 70)
for k in ['auc', 'fpr', 'rc_dana', 'rc_p90']:
    con = np.array(resultados_con[k])
    sin = np.array(resultados_sin[k])
    stat, p = stats.wilcoxon(con, sin)
    print(f"{k:10s} CON ia_racha: {con.mean():.4f} ± {con.std(ddof=1):.4f}  |  "
          f"SIN ia_racha: {sin.mean():.4f} ± {sin.std(ddof=1):.4f}  |  "
          f"Wilcoxon p={p:.4f}")

torch.save({'con': resultados_con, 'sin': resultados_sin}, 'ablacion_ia_racha_resultados.pt')
print("\n[v] Guardado en ablacion_ia_racha_resultados.pt")

In [ ]:
import torch

# 1. Define la ruta de tu archivo
ruta = '/content/resultados/checkpoints/07_metricas_ruido.pt'

# 2. Carga el contenido
datos = torch.load(ruta, weights_only=False)

# 3. Iterar y mostrar TODOS los valores individuales
if isinstance(datos, dict):
    print("=== TODOS LOS VALORES INDIVIDUALES (15 SEMILLAS) ===\n")

    for nombre_modelo, metricas in datos.items():
        print(f"▶ MODELO: {nombre_modelo}")
        print("-" * 80)

        if isinstance(metricas, dict):
            for nombre_metrica, dict_valores in metricas.items():
                if isinstance(dict_valores, dict) and 'valores' in dict_valores:
                    lista_valores = dict_valores['valores']

                    # Formatear cada número de la lista a 4 decimales para que se vea limpio
                    valores_formateados = [f"{v:.4f}" for v in lista_valores]

                    # Imprimir la métrica y su lista completa de valores
                    print(f"   {nombre_metrica.upper():<10} | {', '.join(valores_formateados)}")
        print("\n")
else:
    print("El archivo no contiene un diccionario válido.")

In [ ]:
import torch

# 1. Define la ruta de tu archivo
ruta = '/content/control_3layer_e2e_resultados (2).pt'

# 2. Carga el contenido
datos = torch.load(ruta, weights_only=False)

print(datos)

In [ ]:
import torch
chk = torch.load('/content/resultados/checkpoints/09_multisemilla.pt', weights_only=False)
print("Semillas completadas:", sorted(chk['_estado_interno']['semillas_completadas']))
print("Modelos con datos:", [k for k in chk if not k.startswith('_')])
for nm in chk:
    if not nm.startswith('_'):
        print(f"  {nm}: {len(chk[nm]['auc']['valores'])} valores de AUC guardados")

In [ ]:
import torch
import os

# 🎯 CRÍTICO: Forzar precisión de 64 bits antes de hacer nada más
torch.set_default_dtype(torch.float64)

from modelos import entrenar_qml, entrenar_qtl, entrenar_qnn_layerbylayer
from robustez_estadistica import ejecutar_multisemilla_qml, bootstrap_pareado_operativo, veredicto_orden_dial

print("Cargando datos y baselines desde los pickles...")
agg, ds = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
res_base = torch.load('/content/resultados/checkpoints/02_baselines.pt', weights_only=False)
res_multisemilla = torch.load('/content/resultados/checkpoints/09_multisemilla.pt', weights_only=False)

print("Iniciando entrenamiento QML con 15 semillas...")
resumen = ejecutar_multisemilla_qml(
    ds,
    res_base,
    entrenar_qml,
    entrenar_qtl,
    entrenar_qnn_layerbylayer,
    seeds=range(15),
    guardar_en='/content/resultados/checkpoints/09_multisemilla.pt',
    descargar_cada_semilla=True,
    frecuencia_descarga=2,  # 🎯 Descarga cada 2 semillas para evitar el bloqueo del navegador
)

In [ ]:
import torch
torch.set_default_dtype(torch.float64)

from modelos import entrenar_lstm_standalone_semilla
from robustez_estadistica import backfill_lstm_standalone

ds = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)[1]

# ⚠️ Haz una copia de seguridad primero:
import shutil
shutil.copy('/content/resultados/checkpoints/09_multisemilla.pt',
            '/content/resultados/checkpoints/09_multisemilla_BACKUP.pt')

resumen = backfill_lstm_standalone(
    ds,
    entrenar_lstm_standalone_semilla,
    guardar_en='/content/resultados/checkpoints/09_multisemilla.pt',
)

print("\nAUC Standalone LSTM (backfilled, paired con las semillas cuánticas):")
print(resumen['Standalone LSTM']['auc']['valores'])
print("FPR Standalone LSTM (backfilled):")
print(resumen['Standalone LSTM']['fpr']['valores'])

In [ ]:
import torch
import numpy as np

resumen_gradiente = torch.load('/content/resultados/checkpoints/09_multisemilla.pt', weights_only=False)
resumen_clasico = torch.load('/content/resultados/checkpoints/10_multisemilla_clasico.pt', weights_only=False)  # ajusta ruta

auc_paired = np.array(resumen_gradiente['Standalone LSTM']['auc']['valores'])
auc_independent = np.array(resumen_clasico['LSTM pura']['auc']['valores'])

print("Máxima diferencia absoluta AUC:", np.max(np.abs(auc_paired - auc_independent)))
print("¿Idénticos hasta 1e-10?", np.allclose(auc_paired, auc_independent, atol=1e-10))

In [ ]:
import torch
import numpy as np

resumen_gradiente = torch.load('/content/resultados/checkpoints/09_multisemilla.pt', weights_only=False)

print(resumen_gradiente)

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import wilcoxon

# Import your existing utilities from modelos.py
from modelos import _loop, _evaluar_pytorch

# Cargar el archivo
loaded = torch.load('/content/resultados/checkpoints/01_data.pt', map_location='cpu', weights_only=False)

# Extraer el dataset si es una tupla o lista
if isinstance(loaded, (tuple, list)):
    ds = next((elem for elem in loaded if hasattr(elem, 'in_dim')), loaded[-1])
else:
    ds = loaded

print(f"Dimensión de entrada detectada: {ds.in_dim}")

class LstmMLP4D(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(1, 8, 2, batch_first=True, dropout=0.2, bidirectional=True)
        self.proj = nn.Sequential(nn.Linear(16 + in_dim, 4), nn.Dropout(0.1))

        # --- THE BOTTLENECK CONTROL ---
        # Replaces the 4-qubit VQC (24 params) with a matched 28-parameter MLP
        # Layer 1 (4->2): 10 params | Layer 2 (2->2): 6 params | Layer 3 (2->4): 12 params
        # Total = 28 parameters
        self.mlp = nn.Sequential(
            nn.Linear(4, 2),
            nn.ReLU(),
            nn.Linear(2, 2),
            nn.ReLU(),
            nn.Linear(2, 4)
        )

        self.norm = nn.LayerNorm(4)
        self.clf = nn.Linear(4, 1)

    def forward(self, x):
        _, (hn, _) = self.lstm(x[:, :12].unsqueeze(-1))
        h = torch.cat([hn[-2], hn[-1]], dim=1)
        st = self.proj(torch.cat([h, x], dim=1))

        # Exactly match the VQC input scaling so the bottleneck is identical upstream
        qi = torch.sigmoid(st) * math.pi

        # Pass through Classical MLP instead of VQC
        mlp_out = self.mlp(qi)

        pr_logits = self.clf(self.norm(mlp_out)).squeeze(-1)
        return pr_logits


def run_bottleneck_experiment(ds, qnn_auc_array):
    """
    ds: Your loaded DatasetResult object (containing X_tr, y_tr, etc.)
    qnn_auc_array: The list of 15 AUCs from your LSTM+QNN execution
                   (needed to calculate the Wilcoxon p-value against this model).
    """
    seeds = range(15)
    mlp_aucs = []

    print(f"Starting 15-seed evaluation for LSTM+MLP-4D (Bottleneck Control)...")

    for seed in seeds:
        # Strict seed pairing (matches your QML runs)
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = LstmMLP4D(ds.in_dim)
        dl = DataLoader(TensorDataset(ds.X_tr, ds.y_tr), batch_size=16, shuffle=True, drop_last=True)

        # Match exactly the LSTM+QNN hyperparameters: epochs=25, lr=0.005, patience=6
        model, _ = _loop(model, dl, ds.X_va, ds.y_va, epochs=25, patience=6, lr=5e-3)

        met, _ = _evaluar_pytorch(model, ds.X_te, ds.y_te.numpy())
        mlp_aucs.append(met['auc'])
        print(f"  Seed {seed} -> AUC: {met['auc']:.4f}")

    # Compute final statistics for Table 7
    mean_auc = np.mean(mlp_aucs)
    std_auc = np.std(mlp_aucs, ddof=1)
    max_auc = np.max(mlp_aucs)

    print("\n" + "="*50)
    print("RESULTS FOR TABLE 7:")
    print("="*50)
    print(f"Mean AUC:  {mean_auc:.4f}")
    print(f"Std. Dev:  {std_auc:.4f}")
    print(f"Max AUC:   {max_auc:.4f}")

    # Compute Wilcoxon paired test against QNN for the text block
    if qnn_auc_array is not None and len(qnn_auc_array) == 15:
        stat, p_val = wilcoxon(mlp_aucs, qnn_auc_array)
        print("\nRESULTS FOR TEXT:")
        print(f"Wilcoxon p-value (LSTM+MLP-4D vs LSTM+QNN): {p_val:.4f}")

    return mlp_aucs

# --- HOW TO RUN IT ---
# Replace this array with the 15 AUC values you previously got for LSTM+QNN
# (You can pull this from the `resultados_multi` dict you saved previously)
qnn_15_seed_aucs = qnn_15_seed_aucs = [
    0.6717441877694963, 0.6639204587550194, 0.663594470046083, 0.6448204839450561,
    0.654496421533036, 0.6807977832767792, 0.6483174537318299, 0.6731074132795944,
    0.6596233348644924, 0.627735712063064, 0.6761005823343755, 0.6530442900114095,
    0.6368782135818751, 0.6543926978529199, 0.6749892571902738]

# Call the function (assuming your dataset is loaded as 'ds')
mlp_15_seed_aucs = run_bottleneck_experiment(ds, qnn_15_seed_aucs)

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import wilcoxon

# Import your existing utilities from modelos.py
from modelos import _loop, _evaluar_pytorch

# Cargar el archivo
loaded = torch.load('/content/resultados/checkpoints/01_data.pt', map_location='cpu', weights_only=False)

# Extraer el dataset si es una tupla o lista
if isinstance(loaded, (tuple, list)):
    ds = next((elem for elem in loaded if hasattr(elem, 'in_dim')), loaded[-1])
else:
    ds = loaded

print(f"Dimensión de entrada detectada: {ds.in_dim}")

class LstmMLP4D(nn.Module):
    """Bottleneck-matched classical control: operates on the identical
    4-dimensional projection as LstmQNN, but replaces the VQC with a
    classical MLP. We do NOT force exact parameter-count parity with the
    VQC (~24 params), since doing so requires a degenerate width-2 hidden
    layer that collapses under dead-ReLU pathology in the majority of
    initializations (see Methods/Discussion). Instead we use a modestly
    wider, standard MLP and report both models' parameter counts for
    transparency."""
    def __init__(self, in_dim: int, hidden: int = 16):
        super().__init__()
        self.lstm = nn.LSTM(1, 8, 2, batch_first=True, dropout=0.2, bidirectional=True)
        self.proj = nn.Sequential(nn.Linear(16 + in_dim, 4), nn.Dropout(0.1))

        self.mlp = nn.Sequential(
            nn.Linear(4, hidden),
            nn.GELU(),                 # smoother, non-zero gradient for negative inputs -> no dead-unit collapse
            nn.LayerNorm(hidden),      # stabilizes activations, especially helpful at small batch size (16)
            nn.Linear(hidden, 4),
        )

        self.norm = nn.LayerNorm(4)
        self.clf = nn.Linear(4, 1)

    def forward(self, x):
        _, (hn, _) = self.lstm(x[:, :12].unsqueeze(-1))
        h = torch.cat([hn[-2], hn[-1]], dim=1)
        st = self.proj(torch.cat([h, x], dim=1))
        qi = torch.sigmoid(st) * math.pi
        mlp_out = self.mlp(qi)
        pr_logits = self.clf(self.norm(mlp_out)).squeeze(-1)
        return pr_logits


def run_bottleneck_experiment(ds, qnn_auc_array):
    """
    ds: Your loaded DatasetResult object (containing X_tr, y_tr, etc.)
    qnn_auc_array: The list of 15 AUCs from your LSTM+QNN execution
                   (needed to calculate the Wilcoxon p-value against this model).
    """
    seeds = range(15)
    mlp_aucs = []

    print(f"Starting 15-seed evaluation for LSTM+MLP-4D (Bottleneck Control)...")

    for seed in seeds:
        # Strict seed pairing (matches your QML runs)
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = LstmMLP4D(ds.in_dim)
        dl = DataLoader(TensorDataset(ds.X_tr, ds.y_tr), batch_size=16, shuffle=True, drop_last=True)

        # Match exactly the LSTM+QNN hyperparameters: epochs=25, lr=0.005, patience=6
        model, _ = _loop(model, dl, ds.X_va, ds.y_va, epochs=25, patience=6, lr=5e-3)

        met, _ = _evaluar_pytorch(model, ds.X_te, ds.y_te.numpy())
        mlp_aucs.append(met['auc'])
        print(f"  Seed {seed} -> AUC: {met['auc']:.4f}")

    # Compute final statistics for Table 7
    mean_auc = np.mean(mlp_aucs)
    std_auc = np.std(mlp_aucs, ddof=1)
    max_auc = np.max(mlp_aucs)

    print("\n" + "="*50)
    print("RESULTS FOR TABLE 7:")
    print("="*50)
    print(f"Mean AUC:  {mean_auc:.4f}")
    print(f"Std. Dev:  {std_auc:.4f}")
    print(f"Max AUC:   {max_auc:.4f}")

    # Compute Wilcoxon paired test against QNN for the text block
    if qnn_auc_array is not None and len(qnn_auc_array) == 15:
        stat, p_val = wilcoxon(mlp_aucs, qnn_auc_array)
        print("\nRESULTS FOR TEXT:")
        print(f"Wilcoxon p-value (LSTM+MLP-4D vs LSTM+QNN): {p_val:.4f}")

    return mlp_aucs

# --- HOW TO RUN IT ---
# Replace this array with the 15 AUC values you previously got for LSTM+QNN
# (You can pull this from the `resultados_multi` dict you saved previously)
qnn_15_seed_aucs = qnn_15_seed_aucs = [
    0.6717441877694963, 0.6639204587550194, 0.663594470046083, 0.6448204839450561,
    0.654496421533036, 0.6807977832767792, 0.6483174537318299, 0.6731074132795944,
    0.6596233348644924, 0.627735712063064, 0.6761005823343755, 0.6530442900114095,
    0.6368782135818751, 0.6543926978529199, 0.6749892571902738]

# Call the function (assuming your dataset is loaded as 'ds')
mlp_15_seed_aucs = run_bottleneck_experiment(ds, qnn_15_seed_aucs)

# Report parameter counts explicitly for the paper's methods/appendix
n_params_mlp = sum(p.numel() for p in LstmMLP4D(ds.in_dim).mlp.parameters())
print(f"LstmMLP4D bottleneck MLP parameters: {n_params_mlp}")  # will be larger than 24 — report honestly

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import wilcoxon

# Import your existing utilities from modelos.py
from modelos import _loop, _evaluar_pytorch

# 1. Load the dataset
loaded = torch.load('/content/resultados/checkpoints/01_data.pt', map_location='cpu', weights_only=False)

# Extract the dataset if it's a tuple or list
if isinstance(loaded, (tuple, list)):
    ds = next((elem for elem in loaded if hasattr(elem, 'in_dim')), loaded[-1])
else:
    ds = loaded

print(f"Input dimension detected: {ds.in_dim}")

# 2. Define the Linear Parity Control (20 parameters)
class LinearBottleneckControl(nn.Module):
    def __init__(self, n_q: int = 4):
        super().__init__()
        self.linear = nn.Linear(n_q, n_q)  # 4*4 + 4 = 20 params

    def forward(self, qi):
        return self.linear(qi)

class LstmLinear4D(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(1, 8, 2, batch_first=True, dropout=0.2, bidirectional=True)
        self.proj = nn.Sequential(nn.Linear(16 + in_dim, 4), nn.Dropout(0.1))
        self.bottleneck = LinearBottleneckControl(n_q=4)
        self.norm = nn.LayerNorm(4)
        self.clf = nn.Linear(4, 1)

    def forward(self, x):
        x = x.float()  # <--- CORRECCIÓN AQUÍ: Convierte la entrada a float32
        _, (hn, _) = self.lstm(x[:, :12].unsqueeze(-1))
        h = torch.cat([hn[-2], hn[-1]], dim=1)
        st = self.proj(torch.cat([h, x], dim=1))
        qi = torch.sigmoid(st) * math.pi
        out = self.bottleneck(qi)
        return self.clf(self.norm(out)).squeeze(-1)

# 3. Define the LeakyReLU Parity Control (28 parameters)
class LstmLeakyMLP4D(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(1, 8, 2, batch_first=True, dropout=0.2, bidirectional=True)
        self.proj = nn.Sequential(nn.Linear(16 + in_dim, 4), nn.Dropout(0.1))

        self.mlp = nn.Sequential(
            nn.Linear(4, 2),
            nn.LeakyReLU(0.1),
            nn.Linear(2, 2),
            nn.LeakyReLU(0.1),
            nn.Linear(2, 4)
        )

        # Apply He initialization explicitly
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity='leaky_relu')

        self.norm = nn.LayerNorm(4)
        self.clf = nn.Linear(4, 1)

    def forward(self, x):
        x = x.float()  # <--- CORRECCIÓN AQUÍ: Convierte la entrada a float32
        _, (hn, _) = self.lstm(x[:, :12].unsqueeze(-1))
        h = torch.cat([hn[-2], hn[-1]], dim=1)
        st = self.proj(torch.cat([h, x], dim=1))
        qi = torch.sigmoid(st) * math.pi
        out = self.mlp(qi)
        return self.clf(self.norm(out)).squeeze(-1)

# 4. Define the execution runner
def run_parity_experiment(ds, qnn_auc_array, model_class, model_name):
    seeds = range(15)
    aucs = []

    print(f"\nStarting 15-seed evaluation for {model_name}...")

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = model_class(ds.in_dim)
        dl = DataLoader(TensorDataset(ds.X_tr, ds.y_tr), batch_size=16, shuffle=True, drop_last=True)

        # Match exactly the LSTM+QNN hyperparameters
        model, _ = _loop(model, dl, ds.X_va, ds.y_va, epochs=25, patience=6, lr=5e-3)
        met, _ = _evaluar_pytorch(model, ds.X_te, ds.y_te.numpy())

        aucs.append(met['auc'])
        print(f"  Seed {seed} -> AUC: {met['auc']:.4f}")

    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs, ddof=1)
    max_auc = np.max(aucs)

    print("\n" + "="*50)
    print(f"RESULTS FOR {model_name}:")
    print("="*50)
    print(f"Mean AUC:  {mean_auc:.4f}")
    print(f"Std. Dev:  {std_auc:.4f}")
    print(f"Max AUC:   {max_auc:.4f}")

    # Compute Wilcoxon paired test against QNN
    if qnn_auc_array is not None and len(qnn_auc_array) == 15:
        stat, p_val = wilcoxon(aucs, qnn_auc_array)
        print(f"Wilcoxon p-value vs LSTM+QNN: {p_val:.4f}")

    return aucs

# 5. Execute both experiments
qnn_15_seed_aucs = [
    0.6717441877694963, 0.6639204587550194, 0.663594470046083, 0.6448204839450561,
    0.654496421533036, 0.6807977832767792, 0.6483174537318299, 0.6731074132795944,
    0.6596233348644924, 0.627735712063064, 0.6761005823343755, 0.6530442900114095,
    0.6368782135818751, 0.6543926978529199, 0.6749892571902738
]

linear_aucs = run_parity_experiment(ds, qnn_15_seed_aucs, LstmLinear4D, "LstmLinear4D (20 params)")
leaky_aucs = run_parity_experiment(ds, qnn_15_seed_aucs, LstmLeakyMLP4D, "LstmLeakyMLP4D (28 params)")

In [ ]:
import torch

# 1. Cargar el entorno de datos
agg, ds = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)

# =========================================================
# CÁLCULO DE LOS TODOs
# =========================================================

# TODO 2: Tamaño del panel
total_station_months = len(agg)

# TODO 1: Registros diarios
total_daily_records = int(total_station_months * 30.4368)

# TODO 3: Total de eventos DANA diagnosticados
# En lugar de adivinar el nombre, buscamos automáticamente cualquier atributo en 'ds'
# que contenga la palabra 'dana' (p. ej. dana_train, flag_dana, dana_test...)
atributos_ds = dir(ds)
atributos_dana = [attr for attr in atributos_ds if 'dana' in attr.lower() and not attr.startswith('__')]

total_danas = 0
for attr in atributos_dana:
    tensor_dana = getattr(ds, attr)
    if hasattr(tensor_dana, 'sum'):  # Verificamos que sea un tensor o array sumable
        total_danas += int(tensor_dana.sum().item())

# =========================================================
# RESULTADOS LISTOS PARA COPIAR EN LATEX
# =========================================================
print("🔍 INSPECCIÓN DEL DATASET:")
print(f"Atributos relacionados con DANA encontrados: {atributos_dana}")
print("-" * 70)

print("🎯 TEXTOS LISTOS PARA PEGAR EN TU PAPER:\n")

print("▶ PARA EL TODO 1 Y 2 (Sección 2.1: Study Area and Dataset):")
print(f"...comprising a combined total of {total_daily_records:,} daily records across the four stations, with zero missing values after applying ETL (Extract, Transform, Load) quality control.")
print(f"The records were aggregated to a monthly frequency, yielding a spatiotemporal panel of {total_station_months:,} station-months.")
print("-" * 70)

if total_danas > 0:
    print("▶ PARA EL TODO 3 (Sección 2.3.4: Observational DANA Flag):")
    print(f"This yields {total_danas} flagged events across the multi-station spatiotemporal panel, consistent with the actual incidence of cold-drop dynamics in the Spanish Levante.")
else:
    print("⚠️ AVISO: No se encontraron los tensores DANA en 'ds'.")
    print(f"Por favor, ejecuta `print(dir(ds))` y dime qué sale para ver el nombre exacto.")

Hasta aquí

In [ ]:
"""
Reemplaza la función `ejecutar_multisemilla_qml` dentro de robustez_estadistica.py
por esta versión. El resto del archivo (bootstrap_pareado_operativo,
veredicto_orden_dial, imports de numpy/time/sklearn al principio) no cambia.
"""

import time
import numpy as np

METRICAS_SEGUIDAS = ['acc', 'f1', 'auc', 'fpr', 'rc_dana', 'rc_p90']


def _construir_resumen(acumulado: dict, tiempos: dict, semillas_completadas: set) -> dict:
    """Construye el dict de resumen (mean/std/min/max por modelo y métrica) a
    partir del estado acumulado. Separado en su propia función porque se
    necesita en dos sitios: en el guardado incremental tras cada semilla, y
    al final del todo — incluyendo el caso en que TODAS las semillas ya
    estaban completadas de una ejecución anterior y el bucle no llega a
    ejecutarse ni una vez.
    """
    resumen = {}
    for nm, metricas in acumulado.items():
        resumen[nm] = {}
        for m, vals in metricas.items():
            arr = np.asarray(vals, dtype=float)
            arr = arr[~np.isnan(arr)]
            if len(arr) > 0:
                resumen[nm][m] = {
                    'mean': float(arr.mean()), 'std': float(arr.std()),
                    'min': float(arr.min()), 'max': float(arr.max()),
                    'valores': arr.tolist(),
                }
    resumen['_tiempos_min'] = {nm: float(np.mean(t)) / 60 for nm, t in tiempos.items() if t}
    resumen['_estado_interno'] = {
        'acumulado': acumulado,
        'tiempos': tiempos,
        'semillas_completadas': semillas_completadas,
    }
    return resumen


def _descargar_en_colab(ruta: str) -> None:
    """Dispara la descarga del archivo en el navegador si estamos en Colab.
    Fuera de Colab (o si el navegador bloquea el popup) no rompe nada, solo
    avisa por consola — el archivo ya está guardado en disco de todas formas."""
    try:
        from google.colab import files
        files.download(ruta)
    except ImportError:
        print(f"  [i] No estás en Colab (o 'google.colab' no está disponible): "
              f"omitiendo descarga automática. El archivo sigue en {ruta}.")
    except Exception as e:
        # Los navegadores bloquean popups de descarga repetidos: esto pasa
        # sobre todo a partir de la 2ª-3ª descarga en la misma sesión. No es
        # un fallo del pipeline — el checkpoint en disco es lo que importa.
        print(f"  [!] No se pudo disparar la descarga automática ({e}). "
              f"El archivo está guardado en {ruta}; descárgalo manualmente "
              f"desde el panel de archivos si lo necesitas ahora.")


def ejecutar_multisemilla_qml(ds, res_base, entrenar_qml, entrenar_qtl,
                                entrenar_qnn_layerbylayer, seeds=range(15),
                                guardar_en: str = None,
                                descargar_cada_semilla: bool = True,
                                frecuencia_descarga: int = 1) -> dict:
    """
    Repite el entrenamiento de LSTM+QNN, LSTM+PI-QNN, LSTM+QTL y PI-QNN-LbL
    con varias semillas. Reanuda automáticamente desde `guardar_en` si ya
    existe (útil en Colab, donde el runtime puede desconectarse a media
    ejecución), guarda progreso tras cada semilla, y opcionalmente dispara
    la descarga del archivo en el navegador tras cada guardado.

    Parámetros nuevos respecto a la versión anterior
    --------------------------------------------------
    descargar_cada_semilla : bool
        Si True (por defecto) y `guardar_en` está definido, intenta
        descargar el checkpoint al navegador tras cada semilla completada.
    frecuencia_descarga : int
        Descarga solo cada N semillas (por defecto 1 = todas). Súbelo si el
        navegador empieza a bloquear popups de descarga repetidos — es un
        problema conocido de Colab a partir de la 3ª-4ª descarga seguida.

    Nota importante sobre la reanudación
    -------------------------------------
    Si `guardar_en` ya existe y TODAS las semillas pedidas en `seeds` ya
    están marcadas como completadas, el bucle de entrenamiento no se
    ejecuta ni una vez — la función simplemente reconstruye y devuelve el
    resumen ya guardado. (En la versión anterior esto causaba un
    NameError porque `resumen` solo se definía dentro del bucle.)
    """
    import torch
    import os
    import gc

    seeds = list(seeds)
    n_seeds = len(seeds)

    acumulado = {}
    tiempos = {}
    semillas_completadas = set()

    # --- 1. LÓGICA DE REANUDACIÓN AUTOMÁTICA ---
    if guardar_en and os.path.exists(guardar_en):
        print(f"  [+] Detectado archivo previo en {guardar_en}. Intentando reanudar...")
        try:
            res_previo = torch.load(guardar_en, weights_only=False)
            if '_estado_interno' in res_previo:
                acumulado = res_previo['_estado_interno']['acumulado']
                tiempos = res_previo['_estado_interno']['tiempos']
                semillas_completadas = set(res_previo['_estado_interno']['semillas_completadas'])
                print(f"  [+] Reanudando. {len(semillas_completadas)} semillas ya completadas.")
        except Exception as e:
            print(f"  [!] No se pudo cargar el progreso previo: {e}")

    tiene_qtl = '_modelo_lstm' in res_base
    if not tiene_qtl:
        print("  [!] res_base no tiene '_modelo_lstm' — QTL se omitirá en todas las semillas.")

    for i, seed in enumerate(seeds):
        if seed in semillas_completadas:
            print(f"\n  ── Semilla {seed} ({i + 1}/{n_seeds}) ── [YA COMPLETADA, SALTANDO]")
            continue

        print(f"\n  ── Semilla {seed} ({i + 1}/{n_seeds}) ──")

        # --- LSTM+QNN y LSTM+PI-QNN ---
        torch.manual_seed(seed); np.random.seed(seed)
        t0 = time.time()
        res_qml = entrenar_qml(ds)
        dt_qml = time.time() - t0
        for nm in ('LSTM+QNN', 'LSTM+PI-QNN'):
            if nm not in res_qml:
                continue
            acumulado.setdefault(nm, {m: [] for m in METRICAS_SEGUIDAS})
            tiempos.setdefault(nm, [])
            for m in METRICAS_SEGUIDAS:
                v = res_qml[nm].get(m)
                acumulado[nm][m].append(float(v) if v is not None else float('nan'))
            tiempos[nm].append(dt_qml / 2)

        # --- LSTM+QTL ---
        res_qtl = {}
        if tiene_qtl:
            torch.manual_seed(seed); np.random.seed(seed)
            t0 = time.time()
            res_qtl = entrenar_qtl(ds, res_base['_modelo_lstm'])
            dt_qtl = time.time() - t0
            nm = 'LSTM+QTL'
            if nm in res_qtl:
                acumulado.setdefault(nm, {m: [] for m in METRICAS_SEGUIDAS})
                tiempos.setdefault(nm, [])
                for m in METRICAS_SEGUIDAS:
                    v = res_qtl[nm].get(m)
                    acumulado[nm][m].append(float(v) if v is not None else float('nan'))
                tiempos[nm].append(dt_qtl)

        # --- PI-QNN-LbL ---
        torch.manual_seed(seed); np.random.seed(seed)
        t0 = time.time()
        res_lbl = entrenar_qnn_layerbylayer(ds)
        dt_lbl = time.time() - t0
        nm = 'PI-QNN-LbL'
        if nm in res_lbl:
            acumulado.setdefault(nm, {m: [] for m in METRICAS_SEGUIDAS})
            tiempos.setdefault(nm, [])
            for m in METRICAS_SEGUIDAS:
                v = res_lbl[nm].get(m)
                acumulado[nm][m].append(float(v) if v is not None else float('nan'))
            tiempos[nm].append(dt_lbl)

        # --- Liberar modelos pesados ---
        for res in (res_qml, res_qtl, res_lbl):
            for k in list(res.keys()):
                if isinstance(res[k], dict) and '_modelo' in res[k]:
                    del res[k]['_modelo']

        semillas_completadas.add(seed)

        # --- 2. GUARDADO INCREMENTAL (checkpoint por semilla) ---
        resumen = _construir_resumen(acumulado, tiempos, semillas_completadas)

        if guardar_en:
            torch.save(resumen, guardar_en)
            print(f"  [v] PROGRESO GUARDADO EN DISCO (semillas completadas: "
                  f"{len(semillas_completadas)}/{n_seeds})")

            # --- Descarga automática al navegador (Colab) ---
            if descargar_cada_semilla and (len(semillas_completadas) % frecuencia_descarga == 0):
                _descargar_en_colab(guardar_en)

        # --- 3. Limpieza de memoria agresiva ---
        del res_qml, res_qtl, res_lbl
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --- Reconstrucción final robusta ---
    # No reutiliza el 'resumen' de la última iteración del bucle: si todas las
    # semillas ya estaban completadas (reanudación total, bucle sin
    # iteraciones), esto sigue funcionando porque se reconstruye directamente
    # desde 'acumulado' cargado del checkpoint.
    resumen = _construir_resumen(acumulado, tiempos, semillas_completadas)

    print(f"\n  {'Modelo':<15} {'Métrica':<9} {'Media':>8} {'± Std':>8}   "
          f"[min, max]  (n={len(semillas_completadas)} semillas)")
    print("  " + "-" * 78)
    for nm, metricas in resumen.items():
        if nm.startswith('_'):
            continue
        for m, data in metricas.items():
            if m in ('auc', 'fpr', 'rc_dana'):
                print(f"  {nm:<15} {m:<9} {data['mean']:>8.4f} {data['std']:>8.4f}   "
                      f"[{data['min']:.3f}, {data['max']:.3f}]")

    return resumen

In [ ]:
# ==============================================================================
# CELDA EXTRA: EXPERIMENTO MULTI-SEMILLA Y ESTABILIDAD ESTADÍSTICA
# ¡ADVERTENCIA: ESTA CELDA TARDARÁ VARIAS HORAS EN COMPLETARSE!
# ==============================================================================
from google.colab import files
import torch
import os

# 🎯 FIX: Forzamos precisión de 64 bits para igualar los tensores del dataset
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

from robustez_estadistica import ejecutar_multisemilla_qml, veredicto_orden_dial
from modelos import entrenar_qml, entrenar_qtl, entrenar_qnn_layerbylayer

# 1. Cargar los datos y modelos clásicos ya calculados por el main.py
print("Cargando datos y baselines desde los pickles...")
agg, ds = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
res_base = torch.load('/content/resultados/checkpoints/02_baselines.pt', weights_only=False)

# 2. Ejecutar el entrenamiento con 15 semillas
print("Iniciando entrenamiento QML con 15 semillas...")
resumen_multi = ejecutar_multisemilla_qml(
    ds,
    res_base,
    entrenar_qml,
    entrenar_qtl,
    entrenar_qnn_layerbylayer,
    seeds=range(15),
    guardar_en='/content/resultados/checkpoints/09_multisemilla.pt'
)

# 3. Veredictos de estabilidad (¿Gana siempre o fue suerte?)
print("\nComprobando si el orden entre modelos se mantiene en las 15 semillas:")
veredicto_orden_dial(resumen_multi, metrica='auc', modelos=('LSTM+QNN', 'PI-QNN-LbL'))
veredicto_orden_dial(resumen_multi, metrica='fpr', modelos=('LSTM+QNN', 'PI-QNN-LbL'))
veredicto_orden_dial(resumen_multi, metrica='rc_dana', modelos=('LSTM+QNN', 'PI-QNN-LbL'))

files.download('/content/resultados/checkpoints/09_multisemilla.pt')

In [ ]:
# ============================================================
# Script 1: PR-AUC a partir de checkpoints ya guardados (CORREGIDO)
# ============================================================
import torch
from sklearn.metrics import average_precision_score

# Cargar el contenido
data_loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)

# --- SOLUCIÓN AL ERROR: Búsqueda dinámica del objeto DatasetResult ---
ds = None
if isinstance(data_loaded, tuple):
    print(f"[i] El checkpoint es una tupla de {len(data_loaded)} elementos. Extrayendo el dataset...")
    for item in data_loaded:
        if hasattr(item, 'y_te'):
            ds = item
            break
elif isinstance(data_loaded, dict):
    for val in data_loaded.values():
        if hasattr(val, 'y_te'):
            ds = val
            break
else:
    ds = data_loaded

if ds is None or not hasattr(ds, 'y_te'):
    raise ValueError("No se pudo encontrar el objeto DatasetResult con el atributo 'y_te' en el checkpoint.")
# ---------------------------------------------------------------------

y_te = ds.y_te.numpy()

def extraer_probs(res: dict) -> dict:
    """Misma lógica que _extraer_probs en modelos.py, reutilizada aquí."""
    out = dict(res.get('_probs', {}))
    for k, v in res.items():
        if k.startswith('_'):
            continue
        if isinstance(v, dict) and '_prob' in v:
            out[k] = v['_prob']
    return out

checkpoints = [
    '/content/resultados/checkpoints/02_baselines.pt',
    '/content/resultados/checkpoints/03_qml.pt',
    '/content/resultados/checkpoints/04_qksvm.pt',
    '/content/resultados/checkpoints/08_validacion.pt'
]

probs_todos = {}
for ckpt in checkpoints:
    try:
        res = torch.load(ckpt, weights_only=False)
        probs_todos.update(extraer_probs(res))
    except FileNotFoundError:
        print(f"[!] {ckpt} no encontrado, saltando.")
    except Exception as e:
        print(f"[!] {ckpt} no tiene formato esperado ({e}), revisa manualmente su estructura.")

print(f"\nModelos con probabilidades disponibles: {list(probs_todos.keys())}\n")

base_rate = y_te.mean()  # línea de "no-skill" para contexto
print(f"Base rate (prevalencia positiva en test): {base_rate:.4f}\n")

resultados_pr = {}
print(f"{'Modelo':<28} {'ROC-AUC':>9} {'PR-AUC':>9}")
print("-" * 50)
for nm, prob in sorted(probs_todos.items()):
    from sklearn.metrics import roc_auc_score
    roc = roc_auc_score(y_te, prob)
    pr = average_precision_score(y_te, prob)
    resultados_pr[nm] = {'roc_auc': roc, 'pr_auc': pr}
    print(f"{nm:<28} {roc:>9.4f} {pr:>9.4f}")

torch.save(resultados_pr, '/content/resultados/checkpoints/pr_auc_resultados.pt')
print("\nGuardado en pr_auc_resultados.pt")

In [ ]:
# ============================================================
# Script 2: Confusión ciudad vs física (CORREGIDO)
# ============================================================
import torch
import numpy as np
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# 1. Cargar el contenido
data_loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)

# --- SOLUCIÓN AL ERROR: Búsqueda dinámica del objeto DatasetResult ---
ds = None
if isinstance(data_loaded, tuple):
    print(f"[i] El checkpoint es una tupla de {len(data_loaded)} elementos. Extrayendo el dataset...")
    for item in data_loaded:
        if hasattr(item, 'feature_names'):
            ds = item
            break
elif isinstance(data_loaded, dict):
    for val in data_loaded.values():
        if hasattr(val, 'feature_names'):
            ds = val
            break
else:
    ds = data_loaded

if ds is None or not hasattr(ds, 'feature_names'):
    raise ValueError("No se pudo encontrar el objeto DatasetResult con el atributo 'feature_names' en el checkpoint.")
# ---------------------------------------------------------------------

def entrenar_xgb_y_devolver_importancias(ds_obj, X_sk, X_all_test, feature_names):
    """Réplica exacta del bloque XGBoost inline de entrenar_baselines()
    parametrizada para aceptar features filtradas."""
    imp = SimpleImputer(strategy='mean')
    X_tr_clean = imp.fit_transform(X_sk)
    X_te_clean = imp.transform(X_all_test)

    n_neg, n_pos = (ds_obj.y_sk == 0).sum(), (ds_obj.y_sk == 1).sum()
    xgb = XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_neg / max(n_pos, 1),
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
    xgb.fit(X_tr_clean, ds_obj.y_sk, verbose=False)
    y_te = ds_obj.y_te.numpy() if hasattr(ds_obj.y_te, 'numpy') else ds_obj.y_te
    p_xgb = xgb.predict_proba(X_te_clean)[:, 1]
    auc = roc_auc_score(y_te, p_xgb)
    importancias = dict(zip(feature_names, xgb.feature_importances_))
    return auc, importancias

# --- CON ciudad (features completas) ---
print("[+] Entrenando modelo CON columnas de ciudad...")
feature_names_full = list(ds.feature_names)
auc_full, importancias_full = entrenar_xgb_y_devolver_importancias(
    ds, ds.X_sk, ds.X_all[ds.s2:], feature_names_full
)

# --- SIN ciudad ---
print("[+] Entrenando modelo SIN columnas de ciudad...")
mask_sin_ciudad = np.array([not nm.startswith('ciudad_') for nm in feature_names_full])
feature_names_sin = [nm for nm, keep in zip(feature_names_full, mask_sin_ciudad) if keep]
X_sk_sin = ds.X_sk[:, mask_sin_ciudad]
X_te_sin = ds.X_all[ds.s2:][:, mask_sin_ciudad]

auc_sin, importancias_sin = entrenar_xgb_y_devolver_importancias(
    ds, X_sk_sin, X_te_sin, feature_names_sin
)

print(f"\nAUC CON ciudad_*: {auc_full:.4f}")
print(f"AUC SIN ciudad_*: {auc_sin:.4f}")
print(f"Δ AUC: {auc_sin - auc_full:+.4f}\n")

def rank(imp_dict):
    ordenado = sorted(imp_dict.items(), key=lambda kv: kv[1], reverse=True)
    return {nm: (i + 1, val) for i, (nm, val) in enumerate(ordenado)}

rank_full, rank_sin = rank(importancias_full), rank(importancias_sin)
variables_interes = ['choque_clausius', 'et0_sum', 'et0_lag3m',
                      'potencial_convectivo', 'dp_std', 'ia_racha', 't_anomaly']

print(f"{'Variable':<22} {'Rank CON':>9} {'Gain CON':>10}   {'Rank SIN':>9} {'Gain SIN':>10}")
print("-" * 68)
for v in variables_interes:
    r_f, g_f = rank_full.get(v, ('N/A', 0.0))
    r_s, g_s = rank_sin.get(v, ('N/A', 0.0))
    print(f"{v:<22} {str(r_f):>9} {g_f:>10.4f}   {str(r_s):>9} {g_s:>10.4f}")

torch.save({'auc_full': auc_full, 'auc_sin': auc_sin,
            'rank_full': rank_full, 'rank_sin': rank_sin}, '/content/resultados/checkpoints/confusor_ciudad_resultados.pt')
print("\n[v] Resultados guardados en confusor_ciudad_resultados.pt")

In [ ]:
# ============================================================
# Script 3: Sensibilidad de lambda_cc en PI-QNN-LbL (OPTIMIZADO v2)
# ============================================================
import torch, math
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from modelos import LstmQNN, _evaluar_pytorch

# 🚨 VOLVEMOS A float64 PARA EVITAR EL CHOQUE CON LA CAPA CUÁNTICA
torch.set_default_dtype(torch.float64)

data_loaded = torch.load('/content/resultados/checkpoints/01_data.pt', weights_only=False)
ds = None
if isinstance(data_loaded, tuple):
    for item in data_loaded:
        if hasattr(item, 'in_dim'): ds = item; break
elif isinstance(data_loaded, dict):
    for val in data_loaded.values():
        if hasattr(val, 'in_dim'): ds = val; break
else:
    ds = data_loaded

if ds is None or not hasattr(ds, 'in_dim'):
    raise ValueError("No se pudo encontrar el DatasetResult.")

# --- PRECOMPUTAR TENSORES ESTÁTICOS ---
# 🚨 Forzamos a double (float64) para que coincidan con el modelo y la capa cuántica
X_tr = ds.X_tr.double()
y_tr = ds.y_tr.double()
X_te = ds.X_te.double()
y_te = ds.y_te.double()

n_neg = (y_tr == 0).sum().double()
n_pos = (y_tr == 1).sum().double()
POS_W = torch.tensor([n_neg / (n_pos + 1e-5)], dtype=torch.float64)

T_ANOMALY_TR = torch.tensor([ds.t_anomaly_mes[m] for m in ds.m_tr], dtype=torch.float64)

train_dataset = TensorDataset(X_tr, y_tr, T_ANOMALY_TR)
dl = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)

def make_physics_loss_lambda(pos_weight, lambda_cc, lambda_mse=0.1):
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    CP = 7.0
    def loss_fn(logits, ytrue, qout, T_anomaly_batch, epoch, total_epochs):
        bce = crit(logits, ytrue)
        probs = torch.sigmoid(logits)
        alpha = min(1.0, epoch / (total_epochs * 0.6))

        cc_capacity = torch.exp((CP / 100.0) * T_anomaly_batch * 10.0)
        L_cc = (torch.relu(probs - 0.6) * torch.relu(1.0 - cc_capacity)).mean()

        energy = torch.relu(qout[:, 0]) + torch.relu(qout[:, 1])
        L_mse = ((energy - torch.relu(qout[:, 3]) - 0.5) ** 2).mean()

        return bce + alpha * (lambda_cc * L_cc + lambda_mse * L_mse)
    return loss_fn

def entrenar_lbl_con_lambda(ds, lambda_cc, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    N_Q, N_L = 4, 3

    # 🚨 Forzamos el modelo a double (float64)
    model = LstmQNN(ds.in_dim, n_q=N_Q, n_l=N_L, physics=True).double()
    model.physics_fn = make_physics_loss_lambda(POS_W, lambda_cc)

    # --- FASE LbL ---
    for capa_activa in range(N_L):
        def _make_mask_hook(capa):
            def _hook(grad):
                mask = torch.zeros_like(grad); mask[capa] = 1.0
                return grad * mask
            return _hook

        handle = model.W.register_hook(_make_mask_hook(capa_activa))
        opt_fase = torch.optim.Adam([model.W], lr=5e-3)

        for ep in range(1, 11):
            model.train()
            for bX, by, bT in dl:
                opt_fase.zero_grad()
                pr, qo = model(bX)
                loss = model.physics_fn(pr, by, qo, bT, ep, 10)
                loss.backward()
                opt_fase.step()
        handle.remove()

    # --- FASE FINE-TUNING ---
    for p in model.parameters():
        p.requires_grad_(True)
    opt_ft = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

    for ep in range(1, 26):
        model.train()
        for bX, by, bT in dl:
            opt_ft.zero_grad()
            pr, qo = model(bX)
            loss = model.physics_fn(pr, by, qo, bT, ep, 25)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt_ft.step()

    met, _ = _evaluar_pytorch(model, X_te, y_te.numpy())
    return met['auc']

# ============================================================
# EJECUCIÓN
# ============================================================
LAMBDA_GRID = [0.1, 0.2, 0.5]
N_SEEDS = 3

resultados = {}
print(f"[+] Iniciando experimento ({len(LAMBDA_GRID)*N_SEEDS} ejecuciones totales)...")

import time
start_time = time.time()

for lam in LAMBDA_GRID:
    print(f"  -> Entrenando para λ_cc={lam}...")
    aucs = []
    for seed in range(N_SEEDS):
        seed_start = time.time()
        auc_val = entrenar_lbl_con_lambda(ds, lam, seed)
        aucs.append(auc_val)
        print(f"     [v] Semilla {seed} terminada en {(time.time()-seed_start)/60:.2f} min. AUC={auc_val:.4f}")

    resultados[lam] = {'mean': float(np.mean(aucs)), 'std': float(np.std(aucs, ddof=1)), 'valores': aucs}
    print(f"  [v] Resultado parcial λ_cc={lam}: AUC={resultados[lam]['mean']:.4f} ± {resultados[lam]['std']:.4f}\n")

print(f"[v] Tiempo total: {(time.time()-start_time)/3600:.2f} horas")
torch.save(resultados, '/content/resultados/checkpoints/lambda_sensibilidad_resultados.pt')

In [ ]:
import torch
import pandas as pd
import numpy as np
from scipy import stats

DATA_PATH = "/content/resultados/checkpoints/01_data.pt"  # ajusta si hace falta

loaded = torch.load(DATA_PATH, weights_only=False)
df_raw = loaded[0].copy()
ds = loaded[1]

print(f"Umbral P60 usado en el pipeline (ds.umbral): {ds.umbral}")
# Verifica que esto coincide con los 28.1 mm reportados en el paper.
# Si ds.umbral no existe o da un valor distinto, dímelo y ajustamos.

# --- Construcción de flags directamente sobre df_raw (sin necesidad de alinear con ds) ---
df_raw['P60_exceeded'] = df_raw['precip'] > ds.umbral
df_raw['is_dana_month'] = df_raw['dana_flag'].astype(bool)
df_raw['top_day_share'] = df_raw['precip_max_dia'] / df_raw['precip'].replace(0, np.nan)

p60_months = df_raw[df_raw['P60_exceeded']]

print(f"\nTotal meses en el panel: {len(df_raw)}")
print(f"Meses que superan P60: {len(p60_months)}")
print(f"  de los cuales DANA: {p60_months['is_dana_month'].sum()}")
print(f"  de los cuales No-DANA: {(~p60_months['is_dana_month']).sum()}")

resultados = {}
grupos_share = {}
grupos_extreme_days = {}

for label, mask in [("DANA", p60_months['is_dana_month']),
                     ("No-DANA (P60-exceeding)", ~p60_months['is_dana_month'])]:
    sub = p60_months[mask]

    grupos_share[label] = sub['top_day_share'].dropna().values
    grupos_extreme_days[label] = sub['dias_extremos'].values

    resultados[label] = {
        'n_meses': int(len(sub)),
        'mean_top_day_share': float(sub['top_day_share'].mean()),
        'std_top_day_share': float(sub['top_day_share'].std(ddof=1)),
        'mean_dias_extremos': float(sub['dias_extremos'].mean()),
        'std_dias_extremos': float(sub['dias_extremos'].std(ddof=1)),
    }

# Tests de Mann-Whitney U
u_share, p_share = stats.mannwhitneyu(
    grupos_share["DANA"], grupos_share["No-DANA (P60-exceeding)"], alternative='two-sided'
)
u_days, p_days = stats.mannwhitneyu(
    grupos_extreme_days["DANA"], grupos_extreme_days["No-DANA (P60-exceeding)"], alternative='two-sided'
)

resultados['test_top_day_share'] = {'U': float(u_share), 'p': float(p_share)}
resultados['test_dias_extremos'] = {'U': float(u_days), 'p': float(p_days)}

print("\n=== RESULTADOS ===")
for grupo in ["DANA", "No-DANA (P60-exceeding)"]:
    r = resultados[grupo]
    print(f"\n{grupo} (n={r['n_meses']}):")
    print(f"  top_day_share: {r['mean_top_day_share']:.3f} ± {r['std_top_day_share']:.3f}")
    print(f"  dias_extremos: {r['mean_dias_extremos']:.2f} ± {r['std_dias_extremos']:.2f}")

print(f"\nMann-Whitney U (top_day_share): U={resultados['test_top_day_share']['U']:.1f}, "
      f"p={resultados['test_top_day_share']['p']:.4f}")
print(f"Mann-Whitney U (dias_extremos):  U={resultados['test_dias_extremos']['U']:.1f}, "
      f"p={resultados['test_dias_extremos']['p']:.4f}")